In [ ]:
"""
ML Pipeline for MRoS probabilistic binary precipitation-phase prediction using
raw MRoS point observations as training labels, interpolated predictor surfaces
as gridded features, and leakage-safe LOOCV MRoS support indicators as
point-level predictors.

Scientific intent
-----------------
1. Raw MRoS categorical observations remain the only supervised-learning labels.
2. The supervised target is binary only for observed pure-phase cases:
      rain vs snow.
3. Observed mix is not forced to be a crisp learnable class during fitting.
4. XGBoost outputs rain/snow probabilities, calibrated via isotonic
   regression stratified by wet-bulb temperature regime.
5. A wet-bulb-conditioned Gaussian half-band on calibrated p(snow) is used to
   derive a final 3-class rain / mix / snow product on the full val and test sets.
6. LOOCV-safe MRoS support indicators are retained as predictors only, never as
   labels.

Interpretation
--------------
- p_snow_cal and p_rain_cal are calibrated model-estimated probabilities.
- transition_score summarises ambiguity between rain and snow.
- predicted_mix is derived from the uncertainty band; it is not a directly
  supervised class.
"""

from __future__ import annotations

import itertools
import json
import pickle
import warnings
from pathlib import Path
from typing import Iterable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import xgboost as xgb
from scipy.stats import entropy
from sklearn.utils.class_weight import compute_class_weight

# =============================================================================
# CONFIG
# =============================================================================
REGION = "CA"   # "CA" = Sierra Nevada / Lake Tahoe
                # "CO" = Colorado Mountains

BASE_DIR = Path().resolve().parent
print("BASE_DIR:", BASE_DIR)

ML_DIR = BASE_DIR / "ML_pipeline"
ML_DIR.mkdir(exist_ok=True)
print("ML_DIR:", ML_DIR)

DATA_DIR = Path(r"C:\Users\EmmaGolub\Desktop\MRoS_local\mros-precipitation-phase-product-prototype\outputs")
print("DATA_DIR:", DATA_DIR)

interp_type = "kriging"   # "IDW" or "kriging"

PATHS = {
    "IDW": {
        "interp_grid": DATA_DIR / f"interpolated/{REGION}/IDW_refactored/hourly_predictors_1km_IDW.nc",
        "mros_loocv":  DATA_DIR / f"interpolated/{REGION}/IDW_refactored/mros_loocv_point_predictions_IDW.parquet",
    },
    "kriging": {
        "interp_grid": DATA_DIR / f"interpolated/{REGION}/indicator_kriging_refactored/hourly_predictors_1km_indicator_kriging.nc",
        "mros_loocv":  DATA_DIR / f"interpolated/{REGION}/indicator_kriging_refactored/mros_loocv_point_predictions_kriging.parquet",
    },
    "imerg": DATA_DIR / f"resampled_grids/{REGION}/imerg_hourly_1km.nc",
    # "prism": DATA_DIR / f"resampled_grids/prism_hourly_1km.nc",
}

if interp_type not in {"IDW", "kriging"}:
    raise ValueError(f"Unknown interp_type: {interp_type}")

interp_type_folder = (
    "results_binaryXGB_withIDW_v2" if interp_type == "IDW"
    else "results_binaryXGB_withKriging_v2"
)

SETUP_DIR = DATA_DIR / f"ML_pipeline/compiled_input_predictors/{REGION}" / interp_type_folder
SETUP_DIR.mkdir(parents=True, exist_ok=True)
print("SETUP_DIR:", SETUP_DIR)

MODEL_OUT_DIR = DATA_DIR / f"ML_pipeline/model_artifacts/{REGION}" / interp_type_folder
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)
print("MODEL_OUT_DIR:", MODEL_OUT_DIR)

GRAPHICS_DIR = MODEL_OUT_DIR / "graphics"
GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)
print("GRAPHICS_DIR:", GRAPHICS_DIR)


# Train / val / test proportions
TRAIN_FRAC        = 0.70
VAL_FRAC          = 0.15
# TEST_FRAC is implied as 1 - TRAIN_FRAC - VAL_FRAC
RANDOM_SEED       = 42
N_CAL_FOLDS = 10

# ── model training ────────────────────────────────────────────────────────────
EARLY_STOPPING_ROUNDS = 50
NUM_BOOST_ROUND       = 2000

# ── phase codes / label mapping ───────────────────────────────────────────────
SNOW_CODE          = 0
RAIN_CODE          = 1
MIX_CODE           = 2
BINARY_LABEL_MAP   = {RAIN_CODE: 0, SNOW_CODE: 1}   # 0 = rain, 1 = snow
BINARY_LABEL_NAMES = ["rain", "snow"]
FULL_CLASS_NAMES   = ["snow", "rain", "mix"]

BAND_BASE  = 0.20   # base_half_band used in Gaussian derivation
BAND_EXTRA = 0.15   # extra_half_band
BAND_SIGMA = 2.0    # sigma (°C)

# ── (1) objective-driven weighting ────────────────────────────────────────────
SCALE_POS_WEIGHT_GRID = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.75, 1.0, 1.25, 1.5, 2.0]

# ── (2) calibration  ─────────────────
# Wet-bulb boundary separating near-freezing from clear-phase regimes.
# Calibration models are fitted separately for |Twet| ≤ and > this value.
CLEAR_PHASE_TWET_C    = 2.0   # °C; consistent with Sims & Liu (2015) uncertainty range

# ── (3) wet-bulb-conditioned Gaussian half-band ───────────────────────────────
# half_band(Twet) = base_half_band + extra_half_band * exp(-Twet² / 2σ²)
# rain_thresh = 0.5 - half_band  |  snow_thresh = 0.5 + half_band
BASE_HALF_BAND_GRID  = (0.05, 0.10, 0.15, 0.20)
EXTRA_HALF_BAND_GRID = (0.10, 0.15, 0.20, 0.25, 0.30)
MAX_TOTAL_HALF_BAND  = 0.40
SIGMA_GRID           = (1.0, 1.5, 2.0)   # bounded by Sims & Liu empirical range
MIN_PURE_COVERAGE    = 0.50
MIN_NF_PURE_COVERAGE = 0.35
MIX_CAPTURE_WEIGHT   = 0.5

# ── feature toggles ───────────────────────────────────────────────────────────
USE_TEMP_AIR   = True
USE_PRISM_TAIR = False
USE_TEMP_DEW   = True
USE_TEMP_WET   = True
USE_RH         = False
USE_IMERG_PLP  = True
USE_ELEV       = True
USE_MROS_LOOCV = True

# ── optional diagnostics ──────────────────────────────────────────────────────
RUN_CORR_DIAGNOSTICS = True

In [ ]:
# =============================================================================
# HELPERS
# =============================================================================

from pyproj import Transformer

PHASE_NAME_TO_CODE = {
    "snow": 0,
    "rain": 1,
    "mix": 2,
    "mixed": 2,
    "mixed_phase": 2,
}
PHASE_CODE_TO_NAME = {0: "snow", 1: "rain", 2: "mix"}


def summarize_time(ds: xr.Dataset, name: str) -> None:
    t = pd.to_datetime(ds["time"].values)
    print(f"\n{name} time summary:")
    print("  start:", t.min())
    print("  end:  ", t.max())
    print("  n:", len(t))
    print("  freq guess:", pd.infer_freq(t[: min(10, len(t))]))


def summarize_grid(ds: xr.Dataset, name: str) -> None:
    print(f"\n{name} grid summary:")
    for dim in ["x", "y"]:
        vals = ds[dim].values
        step = np.mean(np.diff(vals)) if len(vals) > 1 else np.nan
        print(f"  {dim}: n={len(vals)}, min={vals.min():.2f}, max={vals.max():.2f}, d≈{step:.2f}")


def candidate_column(df: pd.DataFrame, names: Iterable[str], label: str) -> str:
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"Could not find {label}. Tried: {list(names)}")


def optional_column(df: pd.DataFrame, names: Iterable[str]) -> Optional[str]:
    for n in names:
        if n in df.columns:
            return n
    return None


def normalize_phase_value(v) -> Optional[int]:
    if pd.isna(v):
        return None
    if isinstance(v, str):
        key = v.strip().lower()
        if key in PHASE_NAME_TO_CODE:
            return PHASE_NAME_TO_CODE[key]
        try:
            iv = int(float(key))
            if iv in PHASE_CODE_TO_NAME:
                return iv
        except Exception:
            return None
        return None
    try:
        iv = int(v)
        if iv in PHASE_CODE_TO_NAME:
            return iv
    except Exception:
        return None
    return None


def get_dataset_crs(ds: xr.Dataset, fallback: str = "EPSG:26911"):
    # Prefer rioxarray CRS if present
    try:
        if hasattr(ds, "rio") and ds.rio.crs is not None:
            return ds.rio.crs
    except Exception:
        pass

    # Try attrs as backup
    for key in ["crs", "spatial_ref"]:
        if key in ds.attrs and ds.attrs[key]:
            return ds.attrs[key]

    return fallback


def project_lonlat_to_grid(
    lon: np.ndarray,
    lat: np.ndarray,
    grid_crs,
) -> tuple[np.ndarray, np.ndarray]:
    transformer = Transformer.from_crs("EPSG:4326", grid_crs, always_xy=True)
    x, y = transformer.transform(lon, lat)
    return np.asarray(x, dtype=float), np.asarray(y, dtype=float)


def prep_loocv_table(df: pd.DataFrame, ds_interp: xr.Dataset) -> pd.DataFrame:
    """
    Prepare raw MRoS LOOCV point table for ML.

    Expected refactored-IDW schema from parquet:
      - hour_utc
      - lon, lat
      - elev
      - obs_phase
      - mros_p_snow_loocv, mros_p_mix_loocv, mros_p_rain_loocv
    """
    df = df.copy()

    time_col = candidate_column(df, ["time", "hour_utc", "datetime", "timestamp"], "time column")

    # Refactored IDW LOOCV parquet is lon/lat based; projected x/y may be absent
    x_col = optional_column(df, ["x", "x_proj", "grid_x"])
    y_col = optional_column(df, ["y", "y_proj", "grid_y"])
    lon_col = optional_column(df, ["lon", "longitude", "x_lon"])
    lat_col = optional_column(df, ["lat", "latitude", "y_lat"])

    if (x_col is None or y_col is None) and (lon_col is None or lat_col is None):
        raise KeyError(
            "Could not find projected x/y columns or lon/lat columns in LOOCV parquet."
        )

    obs_col = candidate_column(
        df,
        ["observed_phase", "phase_obs", "raw_phase", "phase", "obs_phase", "phase_label"],
        "observed phase column",
    )
    snow_col = candidate_column(
        df,
        ["mros_p_snow_loocv", "p_snow_loocv", "p_snow_cv", "p_snow"],
        "LOOCV snow probability column",
    )
    mix_col = candidate_column(
        df,
        ["mros_p_mix_loocv", "p_mix_loocv", "p_mix_cv", "p_mix"],
        "LOOCV mix probability column",
    )
    rain_col = candidate_column(
        df,
        ["mros_p_rain_loocv", "p_rain_loocv", "p_rain_cv", "p_rain"],
        "LOOCV rain probability column",
    )

    out = pd.DataFrame({
        "time": pd.to_datetime(df[time_col], errors="coerce", utc=True).dt.floor("h").dt.tz_localize(None),
        "phase_full": df[obs_col].map(normalize_phase_value),
        "mros_p_snow_loocv": pd.to_numeric(df[snow_col], errors="coerce"),
        "mros_p_mix_loocv": pd.to_numeric(df[mix_col], errors="coerce"),
        "mros_p_rain_loocv": pd.to_numeric(df[rain_col], errors="coerce"),
    })

    # Keep lon/lat if available
    if lon_col is not None:
        out["lon"] = pd.to_numeric(df[lon_col], errors="coerce")
    if lat_col is not None:
        out["lat"] = pd.to_numeric(df[lat_col], errors="coerce")

    # Prefer existing projected coordinates if already present
    if x_col is not None and y_col is not None:
        out["x"] = pd.to_numeric(df[x_col], errors="coerce")
        out["y"] = pd.to_numeric(df[y_col], errors="coerce")
    else:
        grid_crs = get_dataset_crs(ds_interp, fallback="EPSG:26911")
        x_proj, y_proj = project_lonlat_to_grid(
            out["lon"].to_numpy(dtype=float),
            out["lat"].to_numpy(dtype=float),
            grid_crs=grid_crs,
        )
        out["x"] = x_proj
        out["y"] = y_proj

    # Preserve optional metadata from IDW LOOCV export
    optional_cols = [
        "hour_utc",
        "elev",
        "pred_phase",
        "obs_p_snow",
        "obs_p_mix",
        "obs_p_rain",
        "pred_max_prob",
        "pred_entropy",
        "pred_correct",
        "station_id",
        "obs_id",
        "source",
    ]
    for c in optional_cols:
        if c in df.columns and c not in out.columns:
            out[c] = df[c]
            
    # Preserve raw observation elevation under a distinct name
    if "elev" in df.columns and "obs_elev" not in out.columns:
        out["obs_elev"] = pd.to_numeric(df["elev"], errors="coerce")

    out["phase_full"] = out["phase_full"].astype("Int64")

    # Renormalize LOOCV probabilities defensively
    probs = out[["mros_p_snow_loocv", "mros_p_mix_loocv", "mros_p_rain_loocv"]].to_numpy(dtype=float)
    probs = np.clip(probs, 0.0, 1.0)
    row_sum = probs.sum(axis=1)
    valid = row_sum > 0
    probs[valid] = probs[valid] / row_sum[valid, None]
    probs[~valid] = np.nan
    out[["mros_p_snow_loocv", "mros_p_mix_loocv", "mros_p_rain_loocv"]] = probs

    out = out.dropna(subset=[
        "time", "x", "y", "phase_full",
        "mros_p_snow_loocv", "mros_p_mix_loocv", "mros_p_rain_loocv",
    ]).copy()

    out["phase_full"] = out["phase_full"].astype(int)
    return out

def build_predictor_cube(
    ds_interp: xr.Dataset,
    ds_imerg: xr.Dataset,
    # ds_prism: xr.Dataset,
    use_temp_air: bool = True,
    use_temp_dew: bool = True,
    use_temp_wet: bool = True,
    use_rh: bool = True,
    use_elev: bool = True,
    use_imerg_plp: bool = True,
    # use_prism_tair: bool = True,
    # use_prism_tmean: bool = False,
    # use_prism_tdmean: bool = False,
    # use_prism_ppt: bool = False,
) -> xr.Dataset:
    """
    Build one aligned predictor cube on common (time, y, x).
    """
    interp_keep = []
    if use_temp_air and "temp_air" in ds_interp:
        interp_keep.append("temp_air")
    if use_temp_dew and "temp_dew" in ds_interp:
        interp_keep.append("temp_dew")
    if use_temp_wet and "temp_wet" in ds_interp:
        interp_keep.append("temp_wet")
    if use_rh and "rh" in ds_interp:
        interp_keep.append("rh")
    if use_elev and "elev" in ds_interp:
        interp_keep.append("elev")

    imerg_keep = []
    if use_imerg_plp and "imerg_plp" in ds_imerg:
        imerg_keep.append("imerg_plp")

    # prism_keep = []
    # if use_prism_tair and "prism_tair" in ds_prism:
    #     prism_keep.append("prism_tair")
    # if use_prism_tmean and "prism_tmean" in ds_prism:
    #     prism_keep.append("prism_tmean")
    # if use_prism_tdmean and "prism_tdmean" in ds_prism:
    #     prism_keep.append("prism_tdmean")
    # if use_prism_ppt and "prism_ppt" in ds_prism:
    #     prism_keep.append("prism_ppt")

    parts = []
    if interp_keep:
        parts.append(ds_interp[interp_keep])
    if imerg_keep:
        parts.append(ds_imerg[imerg_keep])
    # if prism_keep:
    #     parts.append(ds_prism[prism_keep])

    if not parts:
        raise ValueError("No predictor variables selected for predictor cube.")

    ds_pred = xr.merge(parts, compat="override", join="exact")
    return ds_pred

def nearest_index_1d(coord_vals: np.ndarray, query_vals: np.ndarray) -> np.ndarray:
    coord_vals = np.asarray(coord_vals)
    query_vals = np.asarray(query_vals)

    ascending = coord_vals[0] < coord_vals[-1]
    if not ascending:
        coord_work = coord_vals[::-1]
    else:
        coord_work = coord_vals

    idx = np.searchsorted(coord_work, query_vals)
    idx = np.clip(idx, 1, len(coord_work) - 1)

    left = coord_work[idx - 1]
    right = coord_work[idx]
    choose_right = np.abs(query_vals - right) < np.abs(query_vals - left)
    out = idx.copy()
    out[~choose_right] = idx[~choose_right] - 1

    if not ascending:
        out = (len(coord_vals) - 1) - out

    return out.astype(np.int64)


def sample_predictor_cube_to_points_batched(
    points_df: pd.DataFrame,
    ds_pred: xr.Dataset,
    predictor_vars: list[str],
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Sample one aligned predictor cube to raw MRoS observation points,
    processing one hour at a time.

    This only touches hourly slices that are actually present in points_df.
    """
    pts = points_df.copy().reset_index(drop=True)
    pts["time"] = pd.to_datetime(pts["time"]).dt.floor("h")

    # Keep only predictor vars that actually exist
    predictor_vars = [v for v in predictor_vars if v in ds_pred.data_vars]
    if not predictor_vars:
        raise ValueError("No valid predictor_vars found in ds_pred.")

    # Domain filter
    x_min, x_max = float(ds_pred["x"].min()), float(ds_pred["x"].max())
    y_min, y_max = float(ds_pred["y"].min()), float(ds_pred["y"].max())

    in_domain = (
        pts["x"].between(min(x_min, x_max), max(x_min, x_max)) &
        pts["y"].between(min(y_min, y_max), max(y_min, y_max))
    )
    if (~in_domain).any():
        print(f"Dropping {(~in_domain).sum()} rows outside predictor cube extent.")
        pts = pts.loc[in_domain].copy().reset_index(drop=True)

    # Precompute coordinate arrays
    tvals = pd.to_datetime(ds_pred["time"].values)
    xvals = ds_pred["x"].values
    yvals = ds_pred["y"].values

    # Map each point time to nearest time index in ds_pred
    point_time_int = pd.to_datetime(pts["time"]).to_numpy().astype("datetime64[ns]").astype("int64")
    ds_time_int = tvals.astype("datetime64[ns]").astype("int64")
    t_idx = nearest_index_1d(ds_time_int, point_time_int)

    pts["_t_idx"] = t_idx
    pts["_orig_row"] = np.arange(len(pts))

    sampled_chunks = []
    unique_tidx = np.sort(pts["_t_idx"].unique())

    if verbose:
        print(f"Sampling {len(pts)} points across {len(unique_tidx)} hourly slices...")

    for i, tidx in enumerate(unique_tidx, start=1):
        chunk = pts.loc[pts["_t_idx"] == tidx].copy()
        hour_val = tvals[tidx]

        if verbose and (i == 1 or i % 25 == 0 or i == len(unique_tidx)):
            print(f"  Hour batch {i}/{len(unique_tidx)}: {hour_val} ({len(chunk)} points)")

        # nearest x/y indices for this hour's points
        x_idx = nearest_index_1d(xvals, chunk["x"].to_numpy(dtype=float))
        y_idx = nearest_index_1d(yvals, chunk["y"].to_numpy(dtype=float))

        # Read only one 2D slice per variable for this hour
        sampled = pd.DataFrame(index=chunk.index)

        ds_hour = ds_pred.isel(time=int(tidx))

        for v in predictor_vars:
            arr2d = ds_hour[v].values  # shape: (y, x)
            sampled[v] = arr2d[y_idx, x_idx]

        chunk_out = pd.concat([chunk, sampled], axis=1)
        sampled_chunks.append(chunk_out)

    out = pd.concat(sampled_chunks, axis=0).sort_values("_orig_row").reset_index(drop=True)
    out = out.drop(columns=["_t_idx", "_orig_row"])

    return out
def transition_weight(temp_wet: pd.Series, alpha: float) -> np.ndarray:
    tw = temp_wet.to_numpy(dtype=float)
    trans = (tw >= 0.0) & (tw <= 4.0)
    return 1.0 + alpha * trans.astype(float)

In [ ]:
# =============================================================================
# READ DATA
# =============================================================================

interp_grid_path = PATHS[interp_type]["interp_grid"]
loocv_path = PATHS[interp_type]["mros_loocv"]
imerg_path = PATHS["imerg"]
# prism_path = PATHS["prism"]

print("\nOpening datasets...")
ds_interp = xr.open_dataset(interp_grid_path)
ds_imerg = xr.open_dataset(imerg_path)
# ds_prism = xr.open_dataset(prism_path)
df_loocv_raw = pd.read_parquet(loocv_path)

summarize_time(ds_interp, f"{interp_type} interpolation grid")
summarize_time(ds_imerg, "IMERG")
# summarize_time(ds_prism, "PRISM")
summarize_grid(ds_interp, f"{interp_type} interpolation grid")

In [ ]:
# =============================================================================
# TIME SYNCHRONIZATION
# =============================================================================

ds_interp = ds_interp.assign_coords(time=pd.to_datetime(ds_interp.time.values).floor("h"))
ds_imerg  = ds_imerg.assign_coords(time=pd.to_datetime(ds_imerg.time.values).floor("h"))
# ds_prism  = ds_prism.assign_coords(time=pd.to_datetime(ds_prism.time.values).floor("h"))

common_times = np.intersect1d(ds_interp.time.values, ds_imerg.time.values)
# common_times = np.intersect1d(common_times, ds_prism.time.values)

print("Common timesteps across interpolated datasets and IMERG:", len(common_times))
# print("Common timesteps across interpolation, IMERG, and PRISM:", len(common_times))

ds_interp = ds_interp.sel(time=common_times)
ds_imerg  = ds_imerg.sel(time=common_times)
# ds_prism  = ds_prism.sel(time=common_times)

In [ ]:
print("ds_interp x:", ds_interp.x.values[[0, -1]], "n=", len(ds_interp.x))
print("ds_imerg  x:", ds_imerg.x.values[[0, -1]], "n=", len(ds_imerg.x))
print("ds_interp y:", ds_interp.y.values[[0, -1]], "n=", len(ds_interp.y))
print("ds_imerg  y:", ds_imerg.y.values[[0, -1]], "n=", len(ds_imerg.y))
print("ds_interp x dtype:", ds_interp.x.dtype)
print("ds_imerg  x dtype:", ds_imerg.x.dtype)

In [ ]:
# =============================================================================
# BUILD ONE ALIGNED GRIDDED PREDICTOR CUBE
# =============================================================================
# Fix flipped y-axis on IMERG
if ds_imerg.y.values[0] > ds_imerg.y.values[-1]:
    ds_imerg = ds_imerg.isel(y=slice(None, None, -1))
    
ds_pred = build_predictor_cube(
    ds_interp=ds_interp,
    ds_imerg=ds_imerg,
    # ds_prism=ds_prism,
    use_temp_air=USE_TEMP_AIR,
    use_temp_dew=USE_TEMP_DEW,
    use_temp_wet=USE_TEMP_WET,
    use_rh=USE_RH,
    use_elev=USE_ELEV,
    use_imerg_plp=USE_IMERG_PLP,
    # use_prism_tair=USE_PRISM_TAIR,
)

print(ds_pred)
print("Predictor variables in cube:", list(ds_pred.data_vars))

In [ ]:
# =============================================================================
# PREP LOOCV-POINT TABLE (RAW LABELS + LEAKAGE-SAFE MROS PREDICTORS)
# =============================================================================

loocv_df = prep_loocv_table(df_loocv_raw, ds_interp=ds_pred)

print("\nLOOCV point table columns:")
print(loocv_df.columns.tolist())
print("LOOCV class counts:")
print(loocv_df["phase_full"].value_counts(dropna=False).sort_index())

loocv_df = loocv_df[loocv_df["time"].isin(pd.to_datetime(common_times))].copy()
print("Rows after time intersection:", len(loocv_df))


In [ ]:
FEATURES = []

if USE_TEMP_AIR:
    FEATURES.append("temp_air")
# if USE_PRISM_TAIR:
#     FEATURES.append("prism_tair")
if USE_TEMP_DEW:
    FEATURES.append("temp_dew")
if USE_TEMP_WET:
    FEATURES.append("temp_wet")
if USE_RH:
    FEATURES.append("rh")
if USE_IMERG_PLP:
    FEATURES.append("imerg_plp")
if USE_ELEV:
    FEATURES.append("elev")

grid_features = FEATURES.copy()

if USE_MROS_LOOCV:
    FEATURES.extend([
        "mros_p_snow_loocv",
        "mros_p_mix_loocv",
        "mros_p_rain_loocv",
    ])

In [ ]:
# # =============================================================================
# # SAMPLE GRIDDED PREDICTOR CUBE TO RAW MROS OBSERVATION POINTS
# # Comment out if parquet already available to reuse...
# # =============================================================================

# master_df = sample_predictor_cube_to_points_batched(
#     loocv_df,
#     ds_pred=ds_pred,
#     predictor_vars=grid_features,
#     verbose=True,
# )

In [ ]:
# =============================================================================
# BUILD OR RELOAD MASTER ML TABLE
# =============================================================================

REBUILD_MASTER_DF = False  # True = rebuild from current assembled table logic, False = load saved parquet

assembled_parquet = MODEL_OUT_DIR / f"ml_input_points_{interp_type}_binary_uncertainty.parquet"

required_predictors = FEATURES.copy()
required_cols = required_predictors + ["phase_full"]

if REBUILD_MASTER_DF:
    # -------------------------------------------------------------------------
    # Expect master_df to already exist from upstream assembly logic
    # -------------------------------------------------------------------------
    if "master_df" not in globals():
        raise NameError("master_df is not currently defined. Run the upstream assembly cell first.")

    missing_required = [c for c in required_cols if c not in master_df.columns]
    if missing_required:
        raise KeyError(f"Missing required columns after predictor assembly: {missing_required}")

    master_df = master_df.loc[:, ~master_df.columns.duplicated()].copy()
    master_df = master_df.dropna(subset=required_cols).copy()
    master_df["phase_full"] = master_df["phase_full"].astype(int)

    master_df.to_parquet(assembled_parquet, index=False)

    print("Saved assembled pointwise ML table to:", assembled_parquet)

else:
    # -------------------------------------------------------------------------
    # Reload previously saved version
    # -------------------------------------------------------------------------
    if not assembled_parquet.exists():
        raise FileNotFoundError(
            f"Saved master_df parquet not found:\n{assembled_parquet}\n"
            "Set REBUILD_MASTER_DF = True and rerun the upstream assembly first."
        )

    master_df = pd.read_parquet(assembled_parquet)

    missing_required = [c for c in required_cols if c not in master_df.columns]
    if missing_required:
        raise KeyError(f"Missing required columns after reload: {missing_required}")

    master_df = master_df.loc[:, ~master_df.columns.duplicated()].copy()
    master_df = master_df.dropna(subset=required_cols).copy()
    master_df["phase_full"] = master_df["phase_full"].astype(int)

    print("Loaded assembled pointwise ML table from:", assembled_parquet)

print("\nMaster table shape after dropna:", master_df.shape)
print("Class balance before blocking:")
print(master_df["phase_full"].value_counts(normalize=True).sort_index())

# =============================================================================
# TEMPORAL STRUCTURE DIAGNOSTIC
# =============================================================================

obs_times = pd.to_datetime(master_df["time"]).sort_values().reset_index(drop=True)

print("=== Observation time range ===")
print(f"  First: {obs_times.iloc[0]}")
print(f"  Last:  {obs_times.iloc[-1]}")
print(f"  Total: {len(obs_times)} observations")
print(f"  Span:  {(obs_times.iloc[-1] - obs_times.iloc[0]).days} days")

# Observations per day
obs_per_day = (
    master_df.assign(date=pd.to_datetime(master_df["time"]).dt.date)
    .groupby("date")
    .size()
    .rename("n_obs")
)
print(f"\n=== Daily observation counts ===")
print(f"  Days with any observations: {len(obs_per_day)}")
print(f"  Days with 0 observations:   "
      f"{(obs_times.iloc[-1] - obs_times.iloc[0]).days + 1 - len(obs_per_day)}")
print(f"  Mean obs/day (active days): {obs_per_day.mean():.1f}")
print(f"  Median obs/day:             {obs_per_day.median():.1f}")
print(f"  Max obs/day:                {obs_per_day.max()}")
print(f"  Percentiles (25/75/90/99):  "
      f"{obs_per_day.quantile(.25):.0f} / "
      f"{obs_per_day.quantile(.75):.0f} / "
      f"{obs_per_day.quantile(.90):.0f} / "
      f"{obs_per_day.quantile(.99):.0f}")

# Inter-observation gaps — what does the quiet period structure look like?
gaps_h = obs_times.diff().dt.total_seconds().dropna() / 3600
print(f"\n=== Inter-observation gaps ===")
print(f"  Median gap:   {gaps_h.median():.1f} h")
print(f"  Mean gap:     {gaps_h.mean():.1f} h")
print(f"  Gaps > 12h:   {(gaps_h > 12).sum()}")
print(f"  Gaps > 24h:   {(gaps_h > 24).sum()}")
print(f"  Gaps > 48h:   {(gaps_h > 48).sum()}")
print(f"  Gaps > 7d:    {(gaps_h > 168).sum()}")
print(f"  Max gap:      {gaps_h.max():.1f} h ({gaps_h.max()/24:.1f} days)")

# Phase balance by month — is there seasonal structure worth knowing about?
monthly = (
    master_df.assign(month=pd.to_datetime(master_df["time"]).dt.month)
    .groupby("month")["phase_full"]
    .value_counts(normalize=True)
    .unstack(fill_value=0.0)
    .round(3)
)
monthly.columns = ["snow", "rain", "mix"]
monthly.index = monthly.index.map({
    10:"Oct", 11:"Nov", 12:"Dec", 1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May"
})
print(f"\n=== Phase balance by month ===")
print(monthly.to_string())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

obs_per_day.plot(ax=axes[0], kind="bar", color="steelblue", width=0.8)
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Observations")
axes[0].set_title("Daily observation count")
axes[0].tick_params(axis="x", labelbottom=False)
axes[0].grid(axis="y", alpha=0.3)

axes[1].hist(gaps_h.clip(upper=200), bins=60, color="steelblue", edgecolor="none")
axes[1].axvline(24,  color="tab:orange", ls="--", label="24h")
axes[1].axvline(48,  color="tab:red",    ls="--", label="48h")
axes[1].set_xlabel("Gap between consecutive observations (h, clipped at 200h)")
axes[1].set_ylabel("Count")
axes[1].set_title("Inter-observation gap distribution")
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# RANDOM TRAIN / VAL / TEST SPLIT
# =============================================================================
#
# Scientific justification
# ------------------------
# The binary model learns the relationship between instantaneous meteorological
# state variables (Twet, Tair, dewpoint, RH, elevation, IMERG PLP, PRISM Tair,
# LOOCV MRoS probabilities) and observed precipitation phase.  That relationship
# is physically stable across storm events — a given wet-bulb temperature predicts
# snow with the same probability regardless of the synoptic system producing it.
# A randomised split preserves the overall distribution of meteorological states
# and phase labels across train / val / test, which is the primary requirement
# for unbiased generalisation evaluation of a physics-based feature set.
#
# This approach is consistent with Jennings et al. (2025), who evaluated XGBoost
# on randomly split subsets of their crowdsourced and synoptic precipitation phase
# datasets without storm-event blocking.
#
# Known limitation
# ----------------
# Observations from the same storm event may appear in both train and test,
# which means test performance may be mildly optimistic relative to deployment
# on a genuinely novel future season.  Given the dataset size (6,150 observations
# spanning 82 active days) and sparse, opportunistic observation structure of the
# MRoS citizen science network, more complex event-based blocking would risk
# producing splits too small to support stable model fitting and calibration.
# This limitation is documented in the methodology.
#
# Split proportions: 70% train / 15% val / 15% test (by observation count).
# Fixed random seed for reproducibility.

from sklearn.model_selection import train_test_split

points_ml_df = pd.read_parquet(assembled_parquet)

print("Master table shape:", points_ml_df.shape)
print("\nClass balance:")
print(points_ml_df["phase_full"].value_counts(normalize=True).sort_index().round(4))

# ── Two-step stratified split: train vs (val+test), then val vs test ──────────
# Stratify on phase_full so each split preserves the overall snow/rain/mix ratio.

train_df, valtest_df = train_test_split(
    points_ml_df,
    test_size=1.0 - TRAIN_FRAC,
    random_state=RANDOM_SEED,
    stratify=points_ml_df["phase_full"],
)

# Split the remaining 30% evenly into val and test (each ~15% of total).
val_df, test_df = train_test_split(
    valtest_df,
    test_size=0.5,
    random_state=RANDOM_SEED,
    stratify=valtest_df["phase_full"],
)

train_df = train_df.copy()
val_df   = val_df.copy()
test_df  = test_df.copy()

train_df["split"] = "train"
val_df["split"]   = "val"
test_df["split"]  = "test"

best_blocked_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

# ── Summary ───────────────────────────────────────────────────────────────────
print("\nSplit proportions (rows):")
print(best_blocked_df["split"].value_counts(normalize=True).round(3))

print("\nClass balance by split:")
print(
    best_blocked_df.groupby("split")["phase_full"]
    .value_counts(normalize=True)
    .unstack(fill_value=0.0)
    .round(3)
)

split_parquet = SETUP_DIR / f"ml_input_points_{interp_type}_binary_uncertainty_random_split.parquet"
best_blocked_df.to_parquet(split_parquet, index=False)
blocked_parquet = split_parquet   # alias used by downstream cells
print("\nSaved split table to:", split_parquet)

In [ ]:
# =============================================================================
# SPLIT SUMMARY AND DIAGNOSTICS
# =============================================================================
#
# Verifies that the randomised stratified split preserved the phase balance
# and covers the full range of key predictors in each subset.

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors = {"train": "tab:blue", "val": "tab:orange", "test": "tab:red"}

# ── Phase balance by split ────────────────────────────────────────────────────
phase_balance = (
    best_blocked_df.groupby("split")["phase_full"]
    .value_counts(normalize=True)
    .unstack(fill_value=0.0)
    .rename(columns={0: "snow", 1: "rain", 2: "mix"})
)
phase_balance.plot(kind="bar", ax=axes[0], color=["#1f77b4", "#2ca02c", "#e377c2"],
                   edgecolor="k", linewidth=0.5)
axes[0].set_title("Phase balance by split")
axes[0].set_ylabel("Fraction")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(loc="upper right")
axes[0].grid(axis="y", alpha=0.3)

# ── Wet-bulb temperature distribution ────────────────────────────────────────
for split, grp in best_blocked_df.groupby("split"):
    axes[1].hist(grp["temp_wet"].dropna(), bins=30, alpha=0.5,
                 color=colors[split], label=split, density=True)
axes[1].set_xlabel("Wet-bulb temperature (°C)")
axes[1].set_ylabel("Density")
axes[1].set_title("Twet distribution by split\n(thermodynamic coverage check)")
axes[1].legend()
axes[1].grid(alpha=0.3)

# ── Elevation distribution ────────────────────────────────────────────────────
for split, grp in best_blocked_df.groupby("split"):
    axes[2].hist(grp["elev"].dropna(), bins=20, alpha=0.5,
                 color=colors[split], label=split, density=True)
axes[2].set_xlabel("Elevation (m)")
axes[2].set_ylabel("Density")
axes[2].set_title("Elevation distribution by split\n(spatial coverage check)")
axes[2].legend()
axes[2].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(
    GRAPHICS_DIR / f"random_split_diagnostics_{interp_type}.png",
    dpi=200, bbox_inches="tight",
)
plt.close(fig)
print(f"Saved: random_split_diagnostics_{interp_type}.png")

# ── Tabular summary ───────────────────────────────────────────────────────────
print("\nSplit sizes:")
for split in ["train", "val", "test"]:
    sub = best_blocked_df[best_blocked_df["split"] == split]
    print(f"  {split}: {len(sub)} rows  |  "
          f"Twet mean={sub['temp_wet'].mean():.2f}°C  |  "
          f"elev mean={sub['elev'].mean():.0f}m")

In [ ]:
# =============================================================================
# FEATURE MATRICES + OBJECTIVE-DRIVEN WEIGHTING
# =============================================================================

from sklearn.metrics import recall_score

TARGET_FULL      = "phase_full"
PURE_PHASE_CODES = [SNOW_CODE, RAIN_CODE]

train_df_full = best_blocked_df[best_blocked_df["split"] == "train"].copy()
val_df_full   = best_blocked_df[best_blocked_df["split"] == "val"].copy()
test_df_full  = best_blocked_df[best_blocked_df["split"] == "test"].copy()

train_df_fit = train_df_full[train_df_full[TARGET_FULL].isin(PURE_PHASE_CODES)].copy()
val_df_fit   = val_df_full[val_df_full[TARGET_FULL].isin(PURE_PHASE_CODES)].copy()
test_df_fit  = test_df_full[test_df_full[TARGET_FULL].isin(PURE_PHASE_CODES)].copy()

for name, df_ in [("train", train_df_fit), ("val", val_df_fit), ("test", test_df_fit)]:
    if df_.empty:
        raise ValueError(f"{name} pure-phase subset is empty. Cannot fit binary model.")

X_train = train_df_fit[FEATURES].copy()
X_val   = val_df_fit[FEATURES].copy()
X_test  = test_df_fit[FEATURES].copy()

X_val_full  = val_df_full[FEATURES].copy()
X_test_full = test_df_full[FEATURES].copy()

y_train = train_df_fit[TARGET_FULL].map(BINARY_LABEL_MAP).astype(int)
y_val   = val_df_fit[TARGET_FULL].map(BINARY_LABEL_MAP).astype(int)
y_test  = test_df_fit[TARGET_FULL].map(BINARY_LABEL_MAP).astype(int)

y_val_full_phase  = val_df_full[TARGET_FULL].astype(int).to_numpy()
y_test_full_phase = test_df_full[TARGET_FULL].astype(int).to_numpy()
y_val_fit_phase   = val_df_fit[TARGET_FULL].astype(int).to_numpy()
y_test_fit_phase  = test_df_fit[TARGET_FULL].astype(int).to_numpy()

print("\nTraining feature columns:")
print(FEATURES)

print("\nPure-phase fitting counts (rain/snow only):")
for name, df_ in [("Train", train_df_fit), ("Val", val_df_fit), ("Test", test_df_fit)]:
    print(f"{name}:\n", df_[TARGET_FULL].value_counts().sort_index())

if RUN_CORR_DIAGNOSTICS:
    corr = train_df_fit[FEATURES].corr(numeric_only=True)
    corr.to_csv(MODEL_OUT_DIR / "predictor_correlation_matrix.csv")

    high_corr = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
            .stack()
            .reset_index()
    )
    high_corr.columns = ["var1", "var2", "correlation"]
    high_corr["abs_correlation"] = high_corr["correlation"].abs()
    high_corr = high_corr.sort_values("abs_correlation", ascending=False)
    high_corr.to_csv(MODEL_OUT_DIR / "predictor_correlation_pairs.csv", index=False)
    print("\nTop correlated predictor pairs:")
    print(high_corr.head(15))


# =============================================================================
# OBJECTIVE-DRIVEN scale_pos_weight SWEEP
# =============================================================================

def sweep_scale_pos_weight(
    X_tr, y_tr, X_vl, y_vl,
    candidates, base_params,
    probe_rounds=200, early_stopping=20, random_seed=42,
):
    from sklearn.metrics import balanced_accuracy_score, f1_score

    dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=list(X_tr.columns))
    dval   = xgb.DMatrix(X_vl, label=y_vl, feature_names=list(X_vl.columns))

    records = []
    for spw in candidates:
        p = {**base_params, "scale_pos_weight": spw, "seed": random_seed}
        probe = xgb.train(
            params=p, dtrain=dtrain, num_boost_round=probe_rounds,
            evals=[(dval, "val")], early_stopping_rounds=early_stopping,
            verbose_eval=False,
        )
        p_snow = probe.predict(dval)
        y_pred = (p_snow >= 0.5).astype(int)

        snow_rec = recall_score(y_vl, y_pred, pos_label=1, zero_division=0)
        rain_rec = recall_score(y_vl, y_pred, pos_label=0, zero_division=0)
        gap      = abs(snow_rec - rain_rec)

        records.append({
            "scale_pos_weight": spw,
            "snow_recall":       round(snow_rec, 4),
            "rain_recall":       round(rain_rec, 4),
            "recall_gap":        round(gap, 4),
            "balanced_accuracy": round(balanced_accuracy_score(y_vl, y_pred), 4),
            "macro_f1":          round(f1_score(y_vl, y_pred, average="macro",
                                                zero_division=0), 4),
        })

    return pd.DataFrame(records).sort_values("recall_gap").reset_index(drop=True)


BASE_XGB_PARAMS = {
    "objective":        "binary:logistic",
    "eval_metric":      "logloss",
    "max_depth":        6,
    "eta":              0.05,
    "subsample":        0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "lambda":           1.0,
    "tree_method":      "hist",
}

print("\n" + "=" * 60)
print("scale_pos_weight sweep (probe rounds = 200)")
print("=" * 60)

spw_results = sweep_scale_pos_weight(
    X_tr=X_train, y_tr=y_train,
    X_vl=X_val,   y_vl=y_val,
    candidates=SCALE_POS_WEIGHT_GRID,
    base_params=BASE_XGB_PARAMS,
    probe_rounds=350, early_stopping=30, random_seed=RANDOM_SEED,
)

spw_agg = (
    spw_results
    .groupby("scale_pos_weight", as_index=False)
    .agg(
        snow_recall=("snow_recall", "mean"),
        rain_recall=("rain_recall", "mean"),
        recall_gap=("recall_gap", "mean"),
        balanced_accuracy=("balanced_accuracy", "mean"),
        macro_f1=("macro_f1", "mean"),
    )
    .sort_values("recall_gap")
    .reset_index(drop=True)
)

BEST_SCALE_POS_WEIGHT = float(spw_agg.iloc[0]["scale_pos_weight"])
# Then plot spw_agg instead of spw_results

print(spw_agg.to_string(index=False))

print(f"\nSelected scale_pos_weight: {BEST_SCALE_POS_WEIGHT}")
print(f"  Snow recall : {spw_agg.iloc[0]['snow_recall']:.4f}")
print(f"  Rain recall : {spw_agg.iloc[0]['rain_recall']:.4f}")
print(f"  Recall gap  : {spw_agg.iloc[0]['recall_gap']:.4f}")

spw_agg.to_csv(MODEL_OUT_DIR / "scale_pos_weight_sweep.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(spw_agg["scale_pos_weight"], spw_agg["snow_recall"],
             marker="o", label="snow recall")
axes[0].plot(spw_agg["scale_pos_weight"], spw_agg["rain_recall"],
             marker="o", label="rain recall")
axes[0].axvline(BEST_SCALE_POS_WEIGHT, ls="--", color="k", alpha=0.6,
                label=f"selected = {BEST_SCALE_POS_WEIGHT}")
axes[0].set_xlabel("scale_pos_weight"); axes[0].set_ylabel("Recall")
axes[0].set_title("Snow / rain recall vs scale_pos_weight")
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(spw_agg["scale_pos_weight"], spw_agg["recall_gap"],
             marker="o", color="tab:red", label="|snow − rain recall|")
axes[1].axvline(BEST_SCALE_POS_WEIGHT, ls="--", color="k", alpha=0.6,
                label=f"selected = {BEST_SCALE_POS_WEIGHT}")
axes[1].set_xlabel("scale_pos_weight"); axes[1].set_ylabel("|Recall gap|")
axes[1].set_title("Recall gap vs scale_pos_weight")
axes[1].grid(alpha=0.3); axes[1].legend()

fig.tight_layout()
fig.savefig(GRAPHICS_DIR / f"scale_pos_weight_sweep_{interp_type}.png",
            dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Sweep plot saved: scale_pos_weight_sweep_{interp_type}.png")

In [ ]:
# =============================================================================
#  FIT FULL MODEL
# =============================================================================

params = {
    **BASE_XGB_PARAMS,
    "scale_pos_weight": BEST_SCALE_POS_WEIGHT,
    "seed": RANDOM_SEED,
}

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=FEATURES)

evals        = [(dtrain, "train"), (dval, "val")]
evals_result = {}

booster = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=evals,
    evals_result=evals_result,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    verbose_eval=50,
)

print("\nBest iteration:", booster.best_iteration)
print("Best validation score:", booster.best_score)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(evals_result["train"]["logloss"], label="train")
ax.plot(evals_result["val"]["logloss"],   label="val")
ax.axvline(
    booster.best_iteration, ls="--", alpha=0.7,
    label=f"best={booster.best_iteration}",
)
ax.set_xlabel("Boosting iteration")
ax.set_ylabel("Binary log loss")
ax.set_title(
    f"XGBoost binary rain/snow logloss ({interp_type})"
    f"  [scale_pos_weight={BEST_SCALE_POS_WEIGHT}]"
)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(
    GRAPHICS_DIR / f"xgb_binary_logloss_{interp_type}.png",
    dpi=200, bbox_inches="tight",
)
plt.close(fig)

In [ ]:
# =============================================================================
# CALIBRATE + DERIVE UNCERTAINTY PRODUCT
# =============================================================================
#
# Replaces the previous K-fold isotonic calibration with beta calibration
# (Kull et al. 2017) fitted directly on the final booster's score outputs.
#
# Why beta calibration instead of isotonic:
# ------------------------------------------
# Isotonic regression is non-parametric and unconstrained. When test scores
# fall outside the range of scores seen during calibration fitting, sklearn's
# out_of_bounds="clip" maps them to the nearest *output* endpoint — which for
# a well-separated binary problem is very close to 0.0 or 1.0. Diagnostic
# results confirmed 237/353 clear-phase and 111/211 near-freezing test
# predictions were pushed to exact 0 or 1, causing calibrated log loss to
# explode (0.658 / 1.075) while Brier score barely changed (0.058 / 0.092).
#
# Beta calibration (3-param "abm" or 2-param "ab") is a parametric family
# over [0,1] that:
#   - is bounded away from 0 and 1 by construction
#   - cannot extrapolate to exact 0/1 on out-of-range inputs
#   - with "ab" parameterisation imposes symmetry around 0.5, which is
#     physically appropriate for the near-freezing regime where there is no
#     a priori asymmetry in the score-to-probability mapping
#
# Fitting strategy:
# -----------------
# Calibrators are fitted on scores produced by the FINAL booster (trained on
# train only) applied to the full train+val pure-phase pool. This ensures the
# calibrators see the same score distribution as the booster will produce at
# inference time, avoiding the train/val score shift that plagued cross-fitted
# isotonic calibration.
#
# Reference: Kull, M., Silva Filho, T. M., & Flach, P. (2017).
#            Beta calibration: a well-founded and easily implemented improvement
#            on logistic calibration for binary classifiers.
#            AISTATS 2017, PMLR 54:623–631.
# =============================================================================

from betacal import BetaCalibration
from sklearn.metrics import (
    brier_score_loss, log_loss, roc_auc_score,
    f1_score, classification_report, confusion_matrix,
)

# =============================================================================
# STEP 1 — FIT BETA CALIBRATORS ON TRAIN+VAL PURE-PHASE POOL
# =============================================================================

print("=" * 60)
print("STEP 1: Beta calibration (regime-stratified, train+val pool)")
print("=" * 60)

# Build train+val pure-phase pool using the final booster's score outputs
trainval_pure = pd.concat(
    [train_df_fit, val_df_fit]
).reset_index(drop=True)

X_trainval    = trainval_pure[FEATURES]
y_trainval    = trainval_pure[TARGET_FULL].map(BINARY_LABEL_MAP).astype(int)
twet_trainval = trainval_pure["temp_wet"].values

# Score the train+val pool with the FINAL booster (not cross-fitted scores)
p_raw_trainval = booster.predict(xgb.DMatrix(X_trainval, feature_names=FEATURES))

# Regime masks
clear_mask_tv = np.abs(twet_trainval) > CLEAR_PHASE_TWET_C
nf_mask_tv    = ~clear_mask_tv

n_clear = int(clear_mask_tv.sum())
n_nf    = int(nf_mask_tv.sum())
print(f"  Train+val pure-phase pool : {len(trainval_pure)} rows")
print(f"  Clear-phase  (|Twet| > {CLEAR_PHASE_TWET_C}°C): {n_clear} rows")
print(f"  Near-freezing (|Twet| ≤ {CLEAR_PHASE_TWET_C}°C): {n_nf} rows")

# ── Clear-phase calibrator ────────────────────────────────────────────────────
# "abm" = full 3-parameter beta family. Appropriate for clear-phase where
# the score distribution is well-separated and may be asymmetric.
cal_clear = BetaCalibration(parameters="abm")
cal_clear.fit(
    p_raw_trainval[clear_mask_tv].reshape(-1, 1),
    y_trainval.values[clear_mask_tv],
)

# ── Near-freezing calibrator ──────────────────────────────────────────────────
# "ab" = 2-parameter symmetric beta calibration. Appropriate for near-freezing
# where scores are compressed near 0.5 and there is no physical reason to
# expect asymmetry in the score-to-probability mapping.
cal_nf = BetaCalibration(parameters="ab")
cal_nf.fit(
    p_raw_trainval[nf_mask_tv].reshape(-1, 1),
    y_trainval.values[nf_mask_tv],
)

print(f"\n  Beta calibration fit complete.")
print(f"  Clear-phase  model: BetaCalibration(parameters='abm')")
print(f"  Near-freezing model: BetaCalibration(parameters='ab')")

# Store calibrators in a dict for export (mirrors old final_cal_models structure)
final_cal_models = {
    "method":       "beta_calibration",
    "clear_phase":  cal_clear,
    "near_freezing": cal_nf,
    "clear_phase_twet_threshold": CLEAR_PHASE_TWET_C,
    "n_clear_phase":   n_clear,
    "n_near_freezing": n_nf,
}


# =============================================================================
# CALIBRATION FUNCTION
# =============================================================================

def calibrate_beta(
    p_raw: np.ndarray,
    twet:  np.ndarray,
) -> np.ndarray:
    """
    Apply regime-stratified beta calibration to raw XGBoost probabilities.

    Parameters
    ----------
    p_raw : array of shape (n,)
        Raw XGBoost p(snow) outputs (sigmoid probabilities).
    twet : array of shape (n,)
        Wet-bulb temperature (°C) for each observation.

    Returns
    -------
    p_cal : array of shape (n,)
        Calibrated p(snow), clipped to [1e-4, 1-1e-4] as a hard safeguard.
        Beta calibration is bounded away from 0/1 by construction, so the
        clip should never activate in practice.
    """
    p_raw = np.asarray(p_raw, dtype=float)
    twet  = np.asarray(twet,  dtype=float)
    out   = np.empty_like(p_raw)

    nf_mask  = np.abs(twet) <= CLEAR_PHASE_TWET_C
    clr_mask = ~nf_mask

    if clr_mask.any():
        out[clr_mask] = cal_clear.predict(p_raw[clr_mask].reshape(-1, 1))
    if nf_mask.any():
        out[nf_mask]  = cal_nf.predict(p_raw[nf_mask].reshape(-1, 1))

    # Defensive clip — beta calibration should never reach exact 0/1
    # but guard against any numerical edge cases
    return np.clip(out, 1e-4, 1 - 1e-4)


# Alias for backward compatibility with downstream cells that call
# apply_regime_calibration(p_raw, twet, final_cal_models, CLEAR_PHASE_TWET_C)
def apply_regime_calibration(
    p_raw:       np.ndarray,
    twet:        np.ndarray,
    cal_models:  dict,
    clear_twet:  float,
) -> np.ndarray:
    """Drop-in replacement for the old isotonic apply_regime_calibration."""
    return calibrate_beta(p_raw, twet)


# =============================================================================
# IN-SAMPLE CALIBRATION DIAGNOSTICS
# =============================================================================

print("\n── In-sample calibration diagnostics (train+val pool) ──────────────────")
p_cal_trainval = calibrate_beta(p_raw_trainval, twet_trainval)

for regime_name, mask in [
    ("clear_phase  ", clear_mask_tv),
    ("near_freezing", nf_mask_tv),
    ("combined     ", np.ones(len(p_raw_trainval), dtype=bool)),
]:
    y_r   = y_trainval.values[mask]
    p_raw_r = p_raw_trainval[mask]
    p_cal_r = p_cal_trainval[mask]
    print(f"  {regime_name}  n={mask.sum():4d}  "
          f"log_loss raw/cal: {log_loss(y_r, p_raw_r):.4f} / {log_loss(y_r, p_cal_r):.4f}  "
          f"Brier raw/cal: {brier_score_loss(y_r, p_raw_r):.4f} / {brier_score_loss(y_r, p_cal_r):.4f}")

# Check for extreme outputs — should be 0 after beta calibration
n_extreme_cal = ((p_cal_trainval < 1e-6) | (p_cal_trainval > 1 - 1e-6)).sum()
print(f"\n  Extreme calibrated outputs (p<1e-6 or p>1-1e-6): {n_extreme_cal}  "
      f"(should be 0 with beta calibration)")


# =============================================================================
# STEP 2 — RAW PROBABILITIES AND MARGINS FOR ALL SPLITS
# =============================================================================

dval_fit   = xgb.DMatrix(X_val,       feature_names=FEATURES)
dtest_fit  = xgb.DMatrix(X_test,      feature_names=FEATURES)
dval_full  = xgb.DMatrix(X_val_full,  feature_names=FEATURES)
dtest_full = xgb.DMatrix(X_test_full, feature_names=FEATURES)

val_fit_prob_raw    = booster.predict(dval_fit)
test_fit_prob_raw   = booster.predict(dtest_fit)
val_full_prob_raw   = booster.predict(dval_full)
test_full_prob_raw  = booster.predict(dtest_full)

val_fit_margin_raw   = booster.predict(dval_fit,   output_margin=True)
test_fit_margin_raw  = booster.predict(dtest_fit,  output_margin=True)
val_full_margin_raw  = booster.predict(dval_full,  output_margin=True)
test_full_margin_raw = booster.predict(dtest_full, output_margin=True)

# ── Apply beta calibration ────────────────────────────────────────────────────
val_fit_prob_cal   = calibrate_beta(val_fit_prob_raw,   val_df_fit["temp_wet"].to_numpy())
test_fit_prob_cal  = calibrate_beta(test_fit_prob_raw,  test_df_fit["temp_wet"].to_numpy())
val_full_prob_cal  = calibrate_beta(val_full_prob_raw,  val_df_full["temp_wet"].to_numpy())
test_full_prob_cal = calibrate_beta(test_full_prob_raw, test_df_full["temp_wet"].to_numpy())

val_fit_p_rain_cal   = 1.0 - val_fit_prob_cal
test_fit_p_rain_cal  = 1.0 - test_fit_prob_cal
val_full_p_rain_cal  = 1.0 - val_full_prob_cal
test_full_p_rain_cal = 1.0 - test_full_prob_cal

# ── Post-calibration extreme-output check on test ────────────────────────────
print("\n── Post-calibration extreme-output check (test set) ────────────────────")
for regime_name, mask, p_raw_r, p_cal_r, y_r in [
    ("clear_phase  ",
     np.abs(test_df_fit["temp_wet"].to_numpy()) > CLEAR_PHASE_TWET_C,
     test_fit_prob_raw, test_fit_prob_cal,
     (test_df_fit[TARGET_FULL].map(BINARY_LABEL_MAP)).to_numpy()),
    ("near_freezing",
     np.abs(test_df_fit["temp_wet"].to_numpy()) <= CLEAR_PHASE_TWET_C,
     test_fit_prob_raw, test_fit_prob_cal,
     (test_df_fit[TARGET_FULL].map(BINARY_LABEL_MAP)).to_numpy()),
]:
    n_extreme = ((p_cal_r[mask] < 1e-6) | (p_cal_r[mask] > 1 - 1e-6)).sum()
    ll_raw = log_loss(y_r[mask], p_raw_r[mask])
    ll_cal = log_loss(y_r[mask], p_cal_r[mask])
    bs_raw = brier_score_loss(y_r[mask], p_raw_r[mask])
    bs_cal = brier_score_loss(y_r[mask], p_cal_r[mask])
    print(f"  {regime_name}  n={mask.sum():3d}  "
          f"log_loss raw/cal: {ll_raw:.4f} / {ll_cal:.4f}  "
          f"Brier raw/cal: {bs_raw:.4f} / {bs_cal:.4f}  "
          f"extreme: {n_extreme}")


# =============================================================================
# HELPERS — WET-BULB-CONDITIONED GAUSSIAN HALF-BAND
# (unchanged from previous version)
# =============================================================================

def gaussian_half_band(
    temp_wet: np.ndarray,
    base_half_band: float,
    extra_half_band: float,
    sigma: float,
) -> np.ndarray:
    t  = np.asarray(temp_wet, dtype=float)
    hb = base_half_band + extra_half_band * np.exp(-(t ** 2) / (2.0 * sigma ** 2))
    return np.clip(hb, 0.0, 0.5)


def classify_phase_gaussian_band(
    p_snow: np.ndarray,
    temp_wet: np.ndarray,
    base_half_band: float,
    extra_half_band: float,
    sigma: float,
) -> np.ndarray:
    p_snow = np.asarray(p_snow, dtype=float)
    hb     = gaussian_half_band(temp_wet, base_half_band, extra_half_band, sigma)
    pred   = np.full(len(p_snow), MIX_CODE, dtype=int)
    pred[p_snow <= 0.5 - hb] = RAIN_CODE
    pred[p_snow >= 0.5 + hb] = SNOW_CODE
    return pred


def optimize_threshold_params(
    p_snow_cal: np.ndarray,
    temp_wet: np.ndarray,
    y_true_phase: np.ndarray,
    base_half_band_grid=BASE_HALF_BAND_GRID,
    extra_half_band_grid=EXTRA_HALF_BAND_GRID,
    sigma_grid=SIGMA_GRID,
    min_pure_coverage=MIN_PURE_COVERAGE,
    mix_capture_weight=MIX_CAPTURE_WEIGHT,
) -> dict:
    p_snow = np.asarray(p_snow_cal,   dtype=float)
    t_wet  = np.asarray(temp_wet,     dtype=float)
    y_true = np.asarray(y_true_phase, dtype=int)

    true_mix  = y_true == MIX_CODE
    true_pure = np.isin(y_true, [SNOW_CODE, RAIN_CODE])

    records = []
    for base_hb, extra_hb, sigma in itertools.product(
        base_half_band_grid, extra_half_band_grid, sigma_grid
    ):
        if base_hb + extra_hb >= MAX_TOTAL_HALF_BAND:
            continue

        pred      = classify_phase_gaussian_band(p_snow, t_wet, base_hb, extra_hb, sigma)
        pred_mix  = pred == MIX_CODE
        pred_conf = pred != MIX_CODE

        mix_capture   = pred_mix[true_mix].mean()  if true_mix.any()  else np.nan
        pure_cov      = pred_conf[true_pure].mean() if true_pure.any() else np.nan
        mask_cp       = true_pure & pred_conf
        pure_conf_acc = (pred[mask_cp] == y_true[mask_cp]).mean() if mask_cp.any() else np.nan

        composite = (
            mix_capture_weight * mix_capture
            + (1.0 - mix_capture_weight) * pure_conf_acc
            if not (np.isnan(mix_capture) or np.isnan(pure_conf_acc)) else np.nan
        )

        feasible    = (not np.isnan(pure_cov)) and (pure_cov >= min_pure_coverage)
        nf_pure     = true_pure & (np.abs(t_wet) <= 1.0)
        nf_cov      = pred_conf[nf_pure].mean() if nf_pure.any() else np.nan
        nf_feasible = (not np.isnan(nf_cov)) and (nf_cov >= MIN_NF_PURE_COVERAGE)
        feasible    = feasible and nf_feasible

        records.append({
            "base_half_band":    base_hb,
            "extra_half_band":   extra_hb,
            "sigma":             sigma,
            "rain_thresh_0C":    round(0.5 - (base_hb + extra_hb), 4),
            "snow_thresh_0C":    round(0.5 + (base_hb + extra_hb), 4),
            "rain_thresh_far":   round(0.5 - base_hb, 4),
            "snow_thresh_far":   round(0.5 + base_hb, 4),
            "mix_capture_rate":  mix_capture,
            "pure_coverage":     pure_cov,
            "pure_conf_accuracy":pure_conf_acc,
            "composite_score":   composite,
            "feasible":          feasible,
        })

    results_df = pd.DataFrame(records).sort_values(
        ["feasible", "composite_score"], ascending=[False, False]
    ).reset_index(drop=True)

    feasible_df = results_df[results_df["feasible"]]
    best = feasible_df.iloc[0] if not feasible_df.empty else results_df.iloc[0]
    if feasible_df.empty:
        warnings.warn("No feasible parameter combination found. "
                      "Returning best unconstrained result.", stacklevel=2)

    print("\n=== THRESHOLD OPTIMISATION RESULTS ===")
    for k in ["base_half_band", "extra_half_band", "sigma",
              "rain_thresh_0C", "snow_thresh_0C", "rain_thresh_far", "snow_thresh_far",
              "mix_capture_rate", "pure_coverage", "pure_conf_accuracy", "composite_score"]:
        print(f"  {k:28s}: {best[k]:.4f}")

    return {
        "base_half_band":     float(best["base_half_band"]),
        "extra_half_band":    float(best["extra_half_band"]),
        "sigma":              float(best["sigma"]),
        "rain_thresh_0C":     float(best["rain_thresh_0C"]),
        "snow_thresh_0C":     float(best["snow_thresh_0C"]),
        "rain_thresh_far":    float(best["rain_thresh_far"]),
        "snow_thresh_far":    float(best["snow_thresh_far"]),
        "composite_score":    float(best["composite_score"]),
        "mix_capture_rate":   float(best["mix_capture_rate"]),
        "pure_conf_accuracy": float(best["pure_conf_accuracy"]),
        "pure_coverage":      float(best["pure_coverage"]),
        "all_results":        results_df,
    }


def transition_score_from_psnow(p_snow: np.ndarray) -> np.ndarray:
    return 1.0 - np.abs(2.0 * np.asarray(p_snow) - 1.0)


# =============================================================================
# STEP 3 — OPTIMISE WET-BULB-CONDITIONED GAUSSIAN HALF-BAND
# (re-run after recalibration — band params will shift with new probabilities)
# =============================================================================

print("\n" + "=" * 60)
print("STEP 3: Gaussian half-band threshold optimisation")
print(f"  Sigma grid: {SIGMA_GRID} °C  (Sims & Liu 2015; Jennings 2025)")
print("=" * 60)

threshold_result = optimize_threshold_params(
    p_snow_cal=val_full_prob_cal,
    temp_wet=val_df_full["temp_wet"].to_numpy(),
    y_true_phase=y_val_full_phase,
)

BEST_BASE_HB  = threshold_result["base_half_band"]
BEST_EXTRA_HB = threshold_result["extra_half_band"]
BEST_SIGMA    = threshold_result["sigma"]

threshold_result["all_results"].to_csv(
    MODEL_OUT_DIR / "threshold_optimisation_all_results.csv", index=False)
threshold_result["all_results"].head(10).to_csv(
    MODEL_OUT_DIR / "threshold_optimisation_top10.csv", index=False)

# Band shape plot
twet_range = np.linspace(-8, 8, 200)
hb_range   = gaussian_half_band(twet_range, BEST_BASE_HB, BEST_EXTRA_HB, BEST_SIGMA)

fig, ax = plt.subplots(figsize=(8, 4))
ax.fill_between(twet_range, 0.5 - hb_range, 0.5 + hb_range,
                alpha=0.25, color="tab:purple", label="mix / uncertain band")
ax.plot(twet_range, 0.5 - hb_range, color="tab:purple", lw=1.5)
ax.plot(twet_range, 0.5 + hb_range, color="tab:purple", lw=1.5)
ax.axhline(0.5, ls=":", color="k", alpha=0.4)
ax.axvline(0.0, ls="--", color="k", alpha=0.4, label="Twet = 0°C")
ax.axvline( CLEAR_PHASE_TWET_C, ls=":", color="gray", alpha=0.6,
            label=f"calibration regime ±{CLEAR_PHASE_TWET_C}°C")
ax.axvline(-CLEAR_PHASE_TWET_C, ls=":", color="gray", alpha=0.6)
ax.set_xlabel("Wet-bulb temperature (°C)")
ax.set_ylabel("p(snow) threshold")
ax.set_title(f"Gaussian half-band  |  base={BEST_BASE_HB:.2f}, "
             f"extra={BEST_EXTRA_HB:.2f}, σ={BEST_SIGMA:.1f}°C")
ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout()
fig.savefig(GRAPHICS_DIR / f"gaussian_halfband_shape_{interp_type}.png",
            dpi=200, bbox_inches="tight")
plt.close(fig)
print("Band shape plot saved.")


# =============================================================================
# STEP 4 — APPLY THRESHOLDS AND ATTACH PREDICTIONS
# =============================================================================

val_fit_pred_binary  = np.where(val_fit_prob_cal  >= 0.5, SNOW_CODE, RAIN_CODE)
test_fit_pred_binary = np.where(test_fit_prob_cal >= 0.5, SNOW_CODE, RAIN_CODE)

for name, p_cal, tw in [
    ("val_full",  val_full_prob_cal,  val_df_full["temp_wet"].to_numpy()),
    ("test_full", test_full_prob_cal, test_df_full["temp_wet"].to_numpy()),
    ("val_fit",   val_fit_prob_cal,   val_df_fit["temp_wet"].to_numpy()),
    ("test_fit",  test_fit_prob_cal,  test_df_fit["temp_wet"].to_numpy()),
]:
    globals()[f"{name}_pred_phase"]       = classify_phase_gaussian_band(
        p_cal, tw, BEST_BASE_HB, BEST_EXTRA_HB, BEST_SIGMA)
    globals()[f"{name}_transition_score"] = transition_score_from_psnow(p_cal)

val_fit_entropy   = entropy(np.vstack([val_fit_p_rain_cal,   val_fit_prob_cal]),   base=2)
test_fit_entropy  = entropy(np.vstack([test_fit_p_rain_cal,  test_fit_prob_cal]),  base=2)
val_full_entropy  = entropy(np.vstack([val_full_p_rain_cal,  val_full_prob_cal]),  base=2)
test_full_entropy = entropy(np.vstack([test_full_p_rain_cal, test_full_prob_cal]), base=2)


# =============================================================================
# QUICK EVALUATION PRINTOUT
# =============================================================================

print("\n=== BINARY EVALUATION (pure-phase test, 0.5 threshold) ===")
print(confusion_matrix(y_test_fit_phase, test_fit_pred_binary,
                        labels=[SNOW_CODE, RAIN_CODE]))
print(classification_report(
    y_test_fit_phase, test_fit_pred_binary,
    labels=[SNOW_CODE, RAIN_CODE],
    target_names=["snow", "rain"], digits=3, zero_division=0,
))

print("\n=== UNCERTAINTY-DERIVED 3-CLASS (full test) ===")
print(confusion_matrix(y_test_full_phase, test_full_pred_phase,
                        labels=[SNOW_CODE, RAIN_CODE, MIX_CODE]))
print(classification_report(
    y_test_full_phase, test_full_pred_phase,
    labels=[SNOW_CODE, RAIN_CODE, MIX_CODE],
    target_names=FULL_CLASS_NAMES, digits=3, zero_division=0,
))

print(f"\nPure-phase test macro F1:         "
      f"{f1_score(y_test_fit_phase, test_fit_pred_binary, average='macro'):.4f}")
print(f"Pure-phase test calibrated log loss: "
      f"{log_loss(y_test.to_numpy(), test_fit_prob_cal, labels=[0, 1]):.4f}")
print(f"Pure-phase test calibrated Brier:    "
      f"{brier_score_loss(y_test.to_numpy(), test_fit_prob_cal):.4f}")


# =============================================================================
# ATTACH PREDICTIONS TO DATAFRAMES
# =============================================================================

def _attach_preds(base_df, p_snow_raw, p_snow_cal, pred_binary05,
                  pred_phase, transition_score, entropy_binary, raw_margin):
    df = base_df.copy()
    df["p_snow_raw"]                   = p_snow_raw
    df["p_snow_cal"]                   = p_snow_cal
    df["p_rain_raw"]                   = 1.0 - p_snow_raw
    df["p_rain_cal"]                   = 1.0 - p_snow_cal
    df["prediction_binary05"]          = pred_binary05
    df["prediction_phase_uncertainty"] = pred_phase
    df["transition_score"]             = transition_score
    df["entropy_binary"]               = entropy_binary
    df["raw_margin"]                   = raw_margin
    df["is_confident"]                 = pred_phase != MIX_CODE
    df["near_freezing_air_2C"]         = df["temp_air"].abs() <= 2.0
    df["near_freezing_wet_2C"]         = df["temp_wet"].abs() <= 2.0
    return df


df_val_fit = _attach_preds(
    val_df_fit, val_fit_prob_raw, val_fit_prob_cal,
    val_fit_pred_binary, val_fit_pred_phase,
    val_fit_transition_score, val_fit_entropy, val_fit_margin_raw)
df_val_fit["phase_binary"] = y_val.values

df_test_fit = _attach_preds(
    test_df_fit, test_fit_prob_raw, test_fit_prob_cal,
    test_fit_pred_binary, test_fit_pred_phase,
    test_fit_transition_score, test_fit_entropy, test_fit_margin_raw)
df_test_fit["phase_binary"] = y_test.values

df_val_full = _attach_preds(
    val_df_full, val_full_prob_raw, val_full_prob_cal,
    np.where(val_full_prob_cal >= 0.5, SNOW_CODE, RAIN_CODE),
    val_full_pred_phase, val_full_transition_score,
    val_full_entropy, val_full_margin_raw)

df_test_full = _attach_preds(
    test_df_full, test_full_prob_raw, test_full_prob_cal,
    np.where(test_full_prob_cal >= 0.5, SNOW_CODE, RAIN_CODE),
    test_full_pred_phase, test_full_transition_score,
    test_full_entropy, test_full_margin_raw)

print("\nDataframes ready: df_val_fit, df_test_fit, df_val_full, df_test_full")

In [ ]:
# =============================================================================
# EXPORT
# =============================================================================

import json
import pickle

# ── Core model artifacts ──────────────────────────────────────────────────────
booster.save_model(MODEL_OUT_DIR / "xgb_binary_phase_model.bin")

# Save beta calibration models
# (replaces isotonic_calibration_models.pkl — filename updated for clarity)
with open(MODEL_OUT_DIR / "beta_calibration_models.pkl", "wb") as f:
    pickle.dump(final_cal_models, f)

with open(MODEL_OUT_DIR / "feature_names.json", "w") as f:
    json.dump(FEATURES, f, indent=2)

with open(MODEL_OUT_DIR / "training_evals_result.json", "w") as f:
    json.dump(evals_result, f, indent=2)

training_summary = {
    "best_iteration":       int(booster.best_iteration),
    "best_score":           float(booster.best_score),
    "num_boost_round":      int(NUM_BOOST_ROUND),
    "early_stopping_rounds":int(EARLY_STOPPING_ROUNDS),
}
with open(MODEL_OUT_DIR / "training_summary.json", "w") as f:
    json.dump(training_summary, f, indent=2)

calibration_summary = {
    "method": "beta_calibration_regime_stratified",
    "description": (
        "Regime-stratified beta calibration (Kull et al. 2017, AISTATS) applied "
        "to raw XGBoost probabilities. Replaces isotonic regression which produced "
        "exact 0/1 outputs for 62% of test observations due to score extrapolation. "
        "Calibrators are fitted on scores from the FINAL booster applied to the "
        "full train+val pure-phase pool (not cross-fitted scores), ensuring the "
        "calibrators see the same score distribution as deployment. "
        "Regime stratification by wet-bulb temperature: clear-phase uses the full "
        "3-parameter beta family ('abm'); near-freezing uses the symmetric 2-parameter "
        "family ('ab'), which is physically appropriate when no asymmetry is expected "
        "in the score-to-probability mapping near the phase boundary."
    ),
    "reference": (
        "Kull, M., Silva Filho, T. M., & Flach, P. (2017). "
        "Beta calibration: a well-founded and easily implemented improvement on "
        "logistic calibration for binary classifiers. AISTATS 2017, PMLR 54:623-631."
    ),
    "fit_subset": "train+val pure-phase observations",
    "input": "raw XGBoost probability p(snow) from final booster",
    "regimes": {
        "clear_phase_threshold_degC": CLEAR_PHASE_TWET_C,
        "clear_phase_model":    "BetaCalibration(parameters='abm')  # 3-param asymmetric",
        "near_freezing_model":  "BetaCalibration(parameters='ab')   # 2-param symmetric",
        "n_clear_phase":        final_cal_models["n_clear_phase"],
        "n_near_freezing":      final_cal_models["n_near_freezing"],
    },
    "why_not_isotonic": (
        "Diagnostic confirmed IsotonicRegression pushed 237/353 clear-phase and "
        "111/211 near-freezing test predictions to exact 0 or 1 (62% of test set). "
        "Brier score was unaffected (0.056/0.092) but calibrated log loss exploded "
        "(0.658/1.075 vs raw 0.207/0.312). Root cause: test scores fell outside the "
        "score range seen during calibration fitting, and out_of_bounds='clip' maps "
        "these to the nearest output endpoint (near 0 or 1 for a well-separated model)."
    ),
    "uncertainty_thresholds": {
        "method": "wet_bulb_conditioned_gaussian_half_band",
        "description": (
            "half_band(Twet) = base_half_band "
            "+ extra_half_band * exp(-Twet^2 / (2*sigma^2)); "
            "rain_thresh = 0.5 - half_band; snow_thresh = 0.5 + half_band. "
            "Parameters re-optimised on full validation set after beta recalibration. "
            "Physical basis: Harder & Pomeroy (2013); Sims & Liu (2015); "
            "Jennings et al. (2018, 2023, 2025)."
        ),
        "base_half_band":       BEST_BASE_HB,
        "extra_half_band":      BEST_EXTRA_HB,
        "sigma_degC":           BEST_SIGMA,
        "rain_thresh_at_0C":    threshold_result["rain_thresh_0C"],
        "snow_thresh_at_0C":    threshold_result["snow_thresh_0C"],
        "rain_thresh_far":      threshold_result["rain_thresh_far"],
        "snow_thresh_far":      threshold_result["snow_thresh_far"],
        "optimisation_objective": (
            f"composite (mix_capture + pure_conf_accuracy, "
            f"weight={MIX_CAPTURE_WEIGHT}/{1-MIX_CAPTURE_WEIGHT})"
        ),
        "min_pure_coverage_constraint": MIN_PURE_COVERAGE,
        "val_mix_capture_rate":  threshold_result["mix_capture_rate"],
        "val_pure_conf_accuracy":threshold_result["pure_conf_accuracy"],
        "val_pure_coverage":     threshold_result["pure_coverage"],
    },
}

with open(MODEL_OUT_DIR / "calibration_summary.json", "w") as f:
    json.dump(calibration_summary, f, indent=2)

# ── Validation exports ────────────────────────────────────────────────────────
X_val.to_parquet(MODEL_OUT_DIR / "X_val_fit.parquet")
pd.Series(y_val, name="phase_binary").to_frame().to_parquet(
    MODEL_OUT_DIR / "y_val_fit_binary.parquet"
)
df_val_fit.to_parquet(MODEL_OUT_DIR / "val_fit_binary_predictions_combined.parquet")
df_val_full.to_parquet(MODEL_OUT_DIR / "val_full_uncertainty_predictions_combined.parquet")

_OUT_COLS = [
    "time", "x", "y", "phase_full", "prediction_binary05",
    "prediction_phase_uncertainty", "near_freezing_air_2C", "near_freezing_wet_2C",
    "p_snow_raw", "p_snow_cal", "p_rain_raw", "p_rain_cal",
    "transition_score", "entropy_binary", "raw_margin", "is_confident",
]

df_val_fit[_OUT_COLS].rename(columns={"phase_full": "phase"}).to_parquet(
    MODEL_OUT_DIR / "val_fit_binary_predictions.parquet"
)
df_val_full[_OUT_COLS].rename(columns={"phase_full": "phase"}).to_parquet(
    MODEL_OUT_DIR / "val_full_uncertainty_predictions.parquet"
)

# ── Test exports ──────────────────────────────────────────────────────────────
X_test.to_parquet(MODEL_OUT_DIR / "X_test_fit.parquet")
pd.Series(y_test, name="phase_binary").to_frame().to_parquet(
    MODEL_OUT_DIR / "y_test_fit_binary.parquet"
)
df_test_fit.to_parquet(MODEL_OUT_DIR / "test_fit_binary_predictions_combined.parquet")
df_test_full.to_parquet(MODEL_OUT_DIR / "test_full_uncertainty_predictions_combined.parquet")

df_test_fit[_OUT_COLS].rename(columns={"phase_full": "phase"}).to_parquet(
    MODEL_OUT_DIR / "test_fit_binary_predictions.parquet"
)
df_test_full[_OUT_COLS].rename(columns={"phase_full": "phase"}).to_parquet(
    MODEL_OUT_DIR / "test_full_uncertainty_predictions.parquet"
)

# ── Threshold optimisation audit trail ───────────────────────────────────────
threshold_result["all_results"].to_csv(
    MODEL_OUT_DIR / "threshold_optimisation_all_results.csv", index=False
)

# ── Full metadata ─────────────────────────────────────────────────────────────
metadata = {
    "version": "v7_beta_calibration",
    "classes_full":   {"0": "snow", "1": "rain", "2": "mix"},
    "classes_binary_fit": {"0": "rain", "1": "snow"},
    "model_type": (
        "XGBoost (binary:logistic) + regime-stratified beta calibration "
        "(Kull et al. 2017) + wet-bulb-conditioned Gaussian half-band uncertainty product"
    ),
    "interpolation_type": interp_type,
    "xgboost_params": params,
    "label_source": "raw MRoS categorical observations",
    "training_target_definition": (
        "Only observed rain/snow used for supervised binary fitting; "
        "observed mix excluded from fitting and retained for downstream "
        "uncertainty evaluation."
    ),
    "calibration": calibration_summary,
    "uncertainty_product": {
        "type": "wet-bulb-conditioned Gaussian half-band on calibrated p(snow)",
        "half_band_formula": (
            "half_band(Twet) = base_half_band "
            "+ extra_half_band * exp(-Twet^2 / (2*sigma^2))"
        ),
        "rain_if":  "p_snow_cal <= 0.5 - half_band(Twet)",
        "mix_if":   "0.5 - half_band(Twet) < p_snow_cal < 0.5 + half_band(Twet)",
        "snow_if":  "p_snow_cal >= 0.5 + half_band(Twet)",
        "references": [
            "Harder & Pomeroy (2013) Hydrol. Process.",
            "Sims & Liu (2015) J. Hydrometeorol.",
            "Jennings et al. (2018) Nat. Commun.",
            "Jennings et al. (2023) Earth Space Sci.",
            "Jennings et al. (2025) Nat. Commun.",
        ],
    },
    "weighting": {
        "method": "objective_driven_scale_pos_weight",
        "description": (
            "scale_pos_weight swept over a grid; value minimising "
            "|snow_recall - rain_recall| on the pure-phase validation subset selected."
        ),
        "scale_pos_weight_grid":    SCALE_POS_WEIGHT_GRID,
        "best_scale_pos_weight":    BEST_SCALE_POS_WEIGHT,
        "sweep_results_csv":        str(MODEL_OUT_DIR / "scale_pos_weight_sweep.csv"),
    },
    "mros_predictor_source": "LOOCV-safe MRoS support indicators at raw observation points",
    "features": FEATURES,
    "training_summary": training_summary,
    "methodology_notes": [
        "No interpolated MRoS surface is used as a label.",
        "Raw MRoS observations provide the supervised target.",
        "Observed mix is not forced as a crisp learnable class during fitting.",
        "LOOCV-safe MRoS snow/mix/rain support indicators are used as predictors only.",
        "Meteorological predictors are sampled from the selected IDW or kriging full-grid product.",
        "Mix/transition is derived from calibrated rain/snow probabilities via a "
        "wet-bulb-conditioned Gaussian half-band rather than trained as a direct class.",
        "Beta calibration replaces isotonic regression: isotonic produced exact 0/1 outputs "
        "for 62% of test observations due to score-range extrapolation. Beta calibration "
        "is bounded away from 0/1 by construction and has no extrapolation pathology.",
    ],
}
with open(MODEL_OUT_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# ── Feature importance ────────────────────────────────────────────────────────
importance_gain   = booster.get_score(importance_type="gain")
importance_weight = booster.get_score(importance_type="weight")

importance_df = pd.DataFrame({
    "feature": FEATURES,
    "gain":    [importance_gain.get(f,   0.0) for f in FEATURES],
    "weight":  [importance_weight.get(f, 0.0) for f in FEATURES],
}).sort_values("gain", ascending=False)

importance_df.to_csv(MODEL_OUT_DIR / "xgb_feature_importance.csv", index=False)

print("\nAll artifacts saved to:", MODEL_OUT_DIR)

In [ ]:
# =============================================================================
# SHAP ANALYSIS — PER-PHASE FEATURE ATTRIBUTION
# =============================================================================
#
# Computes SHAP values for all observations (train + val + test, full set
# including mix) using TreeExplainer on the fitted XGBoost booster.
#
# SHAP values explain the binary model output (log-odds of snow vs rain).
# Three-phase stratification is achieved by slicing the combined SHAP table
# by the uncertainty-derived phase label (prediction_phase_uncertainty):
#   - snow:  confident prediction above snow threshold
#   - rain:  confident prediction below rain threshold
#   - mix:   observation fell inside the Gaussian uncertainty band
#
# This means "mix SHAP values" are specifically the attributions for
# observations where the model was uncertain — i.e. what features drove
# p(snow) into the ambiguous middle range.
#
# Outputs
# -------
#   shap_values_all.parquet    — one row per obs, one col per feature (SHAP)
#                                plus metadata: time, x, y, phase_full,
#                                prediction_phase_uncertainty, p_snow_cal,
#                                temp_wet, elev, split
#   shap_summary_by_phase.csv  — mean |SHAP| per feature per phase
#   shap_mean_by_phase.png     — beeswarm-style mean |SHAP| bar chart
# =============================================================================

import shap

PHASE_COLORS = {
    "snow": "#1f77b4",   # blue
    "rain": "#2ca02c",   # green
    "mix":  "#e377c2",   # pink
}

# ── Build full combined table (all splits, full phase set) ────────────────────
# Attach split labels so downstream analysis can filter by train/val/test

train_full_shap = train_df_full.copy()
train_full_shap["split"] = "train"

val_full_shap = df_val_full.copy()
val_full_shap["split"] = "val"

test_full_shap = df_test_full.copy()
test_full_shap["split"] = "test"

df_all_full = pd.concat(
    [train_full_shap, val_full_shap, test_full_shap],
    ignore_index=True,
)

# Raw XGBoost probabilities for the full combined set
d_all_full = xgb.DMatrix(df_all_full[FEATURES], feature_names=FEATURES)
p_all_full_raw = booster.predict(d_all_full)

# Calibrated probabilities
p_all_full_cal = apply_regime_calibration(
    p_all_full_raw,
    df_all_full["temp_wet"].to_numpy(),
    final_cal_models,
    CLEAR_PHASE_TWET_C,
)

# Uncertainty-derived phase label for every observation
pred_phase_all = classify_phase_gaussian_band(
    p_all_full_cal,
    df_all_full["temp_wet"].to_numpy(),
    BEST_BASE_HB,
    BEST_EXTRA_HB,
    BEST_SIGMA,
)

# ── Compute SHAP values ───────────────────────────────────────────────────────
print("Computing SHAP values (TreeExplainer, probability output)...")

# tree_path_dependent (the SHAP default) only supports model_output="raw".
# To get SHAP values in probability space we must use feature_perturbation=
# "interventional", which requires a background dataset. We use a stratified
# sample of the training set (500 rows) as the background — large enough to
# be representative, small enough to keep runtime reasonable.

N_BACKGROUND = 500

background = (
    df_all_full[df_all_full["split"] == "train"][FEATURES]
    .groupby(df_all_full[df_all_full["split"] == "train"]["phase_full"], group_keys=False)
    .apply(lambda g: g.sample(
        min(len(g), int(N_BACKGROUND * len(g) / len(df_all_full[df_all_full["split"] == "train"]))),
        random_state=RANDOM_SEED,
    ))
    .reset_index(drop=True)
)
print(f"  Background dataset: {len(background)} rows (stratified sample of train split)")

explainer = shap.TreeExplainer(
    booster,
    data=background,
    feature_perturbation="interventional",
    model_output="probability",
)

shap_values = explainer.shap_values(df_all_full[FEATURES])
print(f"  SHAP array shape: {shap_values.shape}")

# ── Build export table ────────────────────────────────────────────────────────
SHAP_META_COLS = [
    "time", "x", "y", "phase_full",
    "temp_wet", "temp_air", "elev",
]

shap_df = df_all_full[SHAP_META_COLS].copy()
shap_df["split"]                          = df_all_full["split"].values
shap_df["p_snow_cal"]                     = p_all_full_cal
shap_df["prediction_phase_uncertainty"]   = pred_phase_all
shap_df["phase_label"] = shap_df["prediction_phase_uncertainty"].map(
    {SNOW_CODE: "snow", RAIN_CODE: "rain", MIX_CODE: "mix"}
)

for i, feat in enumerate(FEATURES):
    shap_df[f"shap_{feat}"] = shap_values[:, i]

shap_df.to_parquet(MODEL_OUT_DIR / "shap_values_all.parquet")
print(f"  Saved: shap_values_all.parquet  ({len(shap_df)} rows)")

# ── Summary: mean |SHAP| per feature per phase ───────────────────────────────
shap_cols = [f"shap_{f}" for f in FEATURES]

phase_shap_summary = (
    shap_df.groupby("phase_label")[shap_cols]
    .apply(lambda df_: df_.abs().mean())
    .T
    .rename(index=lambda c: c.replace("shap_", ""))
    .rename_axis("feature")
    .reset_index()
)

# Also add an "all" column (mean |SHAP| across all phases)
phase_shap_summary["all"] = (
    pd.DataFrame(np.abs(shap_values), columns=FEATURES).mean(axis=0).values
)

phase_shap_summary = phase_shap_summary.sort_values("all", ascending=False)
phase_shap_summary.to_csv(MODEL_OUT_DIR / "shap_summary_by_phase.csv", index=False)
print("  Saved: shap_summary_by_phase.csv")
print(phase_shap_summary.to_string(index=False))

# ── Plot: mean |SHAP| by phase ────────────────────────────────────────────────
phase_plot_cols = [c for c in ["snow", "rain", "mix"] if c in phase_shap_summary.columns]

x = np.arange(len(FEATURES))
width = 0.25
colors = [PHASE_COLORS["snow"], PHASE_COLORS["rain"], PHASE_COLORS["mix"]]

fig, ax = plt.subplots(figsize=(10, 5))

for i, (phase_col, color) in enumerate(zip(phase_plot_cols, colors)):
    vals = phase_shap_summary.set_index("feature").loc[FEATURES, phase_col].values
    ax.bar(x + i * width, vals, width, label=phase_col, color=color, alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(FEATURES, rotation=35, ha="right")
ax.set_ylabel("Mean |SHAP value|  (probability units)")
ax.set_title("Feature importance by predicted phase (mean |SHAP|)")
ax.legend(title="Predicted phase")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(GRAPHICS_DIR / f"shap_mean_by_phase_{interp_type}.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"  Saved: shap_mean_by_phase_{interp_type}.png")

In [ ]:
# =============================================================================
# SHAP ANALYSIS — PER-PHASE FEATURE ATTRIBUTION + WET-BULB BIN STRATIFICATION
# =============================================================================
#
# Extends the original SHAP analysis with wet-bulb temperature binning to
# reveal how feature importance shifts across the freezing transition.
#
# New outputs
# -----------
#   shap_by_wetbulb_bin.csv         — mean |SHAP| per feature per T_wet bin
#   shap_wetbulb_heatmap.png        — heatmap: features × T_wet bins (all obs)
#   shap_wetbulb_heatmap_<phase>.png — same, split by phase (snow/rain/mix)
#   shap_wetbulb_lineplot.png       — line plot: top-N features across T_wet bins
#   shap_beeswarm_nearfreeze.png    — beeswarm for |T_wet| <= 2°C subset
#   shap_beeswarm_clearphase.png    — beeswarm for |T_wet| > 2°C subset
# =============================================================================

import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

# ── Reuse existing objects from original script ───────────────────────────────
# Assumes the following already exist in scope:
#   booster, FEATURES, df_all_full, shap_df, shap_values,
#   PHASE_COLORS, MODEL_OUT_DIR, GRAPHICS_DIR, RANDOM_SEED

PHASE_COLORS = {
    "snow": "#1f77b4",
    "rain": "#2ca02c",
    "mix":  "#e377c2",
}

# ── Wet-bulb bin definitions ──────────────────────────────────────────────────
# 1°C bins from -6 to +6, plus catch-all tails
TWET_BIN_EDGES   = [-np.inf, -6, -5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5, 6, np.inf]
TWET_BIN_LABELS  = [
    "<-6", "-6–-5", "-5–-4", "-4–-3", "-3–-2", "-2–-1",
    "-1–0", "0–1", "1–2", "2–3", "3–4", "4–5", "5–6", ">6"
]
NEAR_FREEZE_THRESH = 2.0   # |T_wet| <= this → "near-freezing" regime
TOP_N_FEATURES     = 5     # how many features to highlight in line plot

shap_cols = [f"shap_{f}" for f in FEATURES]

# Assign bin to every row in shap_df
shap_df["twet_bin"] = pd.cut(
    shap_df["temp_wet"],
    bins=TWET_BIN_EDGES,
    labels=TWET_BIN_LABELS,
    right=True,
)

# ── 1. Mean |SHAP| per feature × T_wet bin (all obs) ─────────────────────────
print("Computing mean |SHAP| by wet-bulb bin...")

shap_by_bin = (
    shap_df.groupby("twet_bin", observed=True)[shap_cols]
    .apply(lambda df_: df_.abs().mean())
    .T
    .rename(index=lambda c: c.replace("shap_", ""))
    .rename_axis("feature")
)

shap_by_bin.to_csv(MODEL_OUT_DIR / "shap_by_wetbulb_bin.csv")
print("  Saved: shap_by_wetbulb_bin.csv")
print(shap_by_bin.to_string())

# ── Helper: plot a single heatmap ─────────────────────────────────────────────
def plot_shap_heatmap(data: pd.DataFrame, title: str, savepath, vmax=None):
    """
    data: DataFrame with features as index, T_wet bins as columns.
    Columns that are entirely NaN (empty bin) are dropped.
    """
    data = data.loc[:, data.notna().any()]  # drop all-NaN columns
    if vmax is None:
        vmax = data.max().max()

    fig, ax = plt.subplots(figsize=(max(10, len(data.columns) * 0.7), len(FEATURES) * 0.55 + 1.5))
    sns.heatmap(
        data,
        ax=ax,
        cmap="YlOrRd",
        vmin=0,
        vmax=vmax,
        annot=True,
        fmt=".3f",
        linewidths=0.4,
        linecolor="#cccccc",
        cbar_kws={"label": "Mean |SHAP| (probability units)"},
    )
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_xlabel("Wet-bulb temperature bin (°C)")
    ax.set_ylabel("Feature")
    # shade the near-freezing columns
    near_freeze_cols = [
        i for i, lbl in enumerate(data.columns)
        if lbl in ["-2–-1", "-1–0", "0–1", "1–2"]
    ]
    for col_i in near_freeze_cols:
        ax.add_patch(plt.Rectangle(
            (col_i, 0), 1, len(data), fill=True,
            color="steelblue", alpha=0.07, zorder=0
        ))
    fig.tight_layout()
    fig.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {savepath.name}")


# ── 2. Heatmap — all observations ─────────────────────────────────────────────
plot_shap_heatmap(
    shap_by_bin,
    "Mean |SHAP| by wet-bulb bin — all observations",
    GRAPHICS_DIR / "shap_wetbulb_heatmap_all.png",
)

# ── 3. Heatmaps — per phase ───────────────────────────────────────────────────
# Use a shared vmax so phases are visually comparable
global_vmax = 0.0
phase_bin_tables = {}

for phase in ["snow", "rain", "mix"]:
    mask = shap_df["phase_label"] == phase
    if mask.sum() == 0:
        continue
    tbl = (
        shap_df[mask]
        .groupby("twet_bin", observed=True)[shap_cols]
        .apply(lambda df_: df_.abs().mean())
        .T
        .rename(index=lambda c: c.replace("shap_", ""))
        .rename_axis("feature")
    )
    phase_bin_tables[phase] = tbl
    global_vmax = max(global_vmax, tbl.max().max())

for phase, tbl in phase_bin_tables.items():
    plot_shap_heatmap(
        tbl,
        f"Mean |SHAP| by wet-bulb bin — predicted phase: {phase}",
        GRAPHICS_DIR / f"shap_wetbulb_heatmap_{phase}.png",
        vmax=global_vmax,
    )

# ── 4. Line plot — top-N features across T_wet bins (all obs) ─────────────────
print("Plotting top-N feature line plot across T_wet bins...")

# Rank features by their mean |SHAP| in the near-freezing bins only
near_freeze_mask_bin = shap_df["twet_bin"].isin(["-2–-1", "-1–0", "0–1", "1–2"])
top_features_nearfreeze = (
    shap_df[near_freeze_mask_bin][shap_cols]
    .abs()
    .mean()
    .rename(index=lambda c: c.replace("shap_", ""))
    .nlargest(TOP_N_FEATURES)
    .index.tolist()
)
print(f"  Top {TOP_N_FEATURES} features near freezing: {top_features_nearfreeze}")

# Use a colormap for the lines
line_colors = plt.cm.tab10(np.linspace(0, 0.9, TOP_N_FEATURES))
valid_bins = [b for b in TWET_BIN_LABELS if b in shap_by_bin.columns]

fig, ax = plt.subplots(figsize=(12, 5))

for feat, color in zip(top_features_nearfreeze, line_colors):
    if feat not in shap_by_bin.index:
        continue
    y = shap_by_bin.loc[feat, valid_bins].values.astype(float)
    ax.plot(valid_bins, y, marker="o", label=feat, color=color, linewidth=1.8)

# shade near-freezing zone
nf_indices = [valid_bins.index(b) for b in ["-2–-1", "-1–0", "0–1", "1–2"] if b in valid_bins]
if nf_indices:
    ax.axvspan(min(nf_indices) - 0.5, max(nf_indices) + 0.5,
               alpha=0.10, color="steelblue", label="|T_wet| ≤ 2°C zone")

ax.set_xlabel("Wet-bulb temperature bin (°C)")
ax.set_ylabel("Mean |SHAP| (probability units)")
ax.set_title(f"Top {TOP_N_FEATURES} features (ranked by near-freezing importance) across T_wet bins")
ax.legend(loc="upper left", fontsize=9, framealpha=0.8)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=35, ha="right")
fig.tight_layout()
fig.savefig(GRAPHICS_DIR / "shap_wetbulb_lineplot.png", dpi=200, bbox_inches="tight")
plt.close(fig)
print("  Saved: shap_wetbulb_lineplot.png")

# ── 5. Beeswarm plots — near-freezing vs. clear-phase regimes ─────────────────
# shap.summary_plot (beeswarm) needs the raw SHAP array + feature matrix,
# so we index back into shap_values using the mask rows.

def plot_beeswarm(mask: np.ndarray, title: str, savepath, max_display=10):
    """
    mask: boolean array aligned with df_all_full / shap_values rows.
    """
    if mask.sum() == 0:
        print(f"  Skipping beeswarm (no observations): {title}")
        return

    sv_subset   = shap_values[mask]
    feat_subset = df_all_full[FEATURES].values[mask]

    fig, ax = plt.subplots(figsize=(8, 5))
    shap.summary_plot(
        sv_subset,
        features=feat_subset,
        feature_names=FEATURES,
        max_display=max_display,
        show=False,
        plot_size=None,
    )
    plt.title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {savepath.name}")


near_freeze_row_mask = np.abs(shap_df["temp_wet"].values) <= NEAR_FREEZE_THRESH
clear_phase_row_mask = np.abs(shap_df["temp_wet"].values) >  NEAR_FREEZE_THRESH

plot_beeswarm(
    near_freeze_row_mask,
    f"SHAP beeswarm — near-freezing regime (|T_wet| ≤ {NEAR_FREEZE_THRESH}°C)",
    GRAPHICS_DIR / "shap_beeswarm_nearfreeze.png",
)
plot_beeswarm(
    clear_phase_row_mask,
    f"SHAP beeswarm — clear-phase regime (|T_wet| > {NEAR_FREEZE_THRESH}°C)",
    GRAPHICS_DIR / "shap_beeswarm_clearphase.png",
)

# ── 6. Bonus: near-freezing beeswarm split by phase ───────────────────────────
for phase in ["snow", "rain", "mix"]:
    mask = near_freeze_row_mask & (shap_df["phase_label"].values == phase)
    plot_beeswarm(
        mask,
        f"SHAP beeswarm — near-freezing, predicted: {phase}  (|T_wet| ≤ {NEAR_FREEZE_THRESH}°C)",
        GRAPHICS_DIR / f"shap_beeswarm_nearfreeze_{phase}.png",
    )

# ── 7. Summary table: near-freeze vs. clear-phase importance ratio ─────────────
# Highlights features that are disproportionately important near freezing
print("\nFeature importance ratio (near-freezing / clear-phase):")

mean_shap_nf = (
    pd.DataFrame(np.abs(shap_values[near_freeze_row_mask]), columns=FEATURES).mean()
)
mean_shap_cp = (
    pd.DataFrame(np.abs(shap_values[clear_phase_row_mask]), columns=FEATURES).mean()
)

ratio_df = pd.DataFrame({
    "mean_shap_near_freeze": mean_shap_nf,
    "mean_shap_clear_phase": mean_shap_cp,
    "ratio_nf_over_cp":      mean_shap_nf / mean_shap_cp.replace(0, np.nan),
}).sort_values("ratio_nf_over_cp", ascending=False)

print(ratio_df.to_string())
ratio_df.to_csv(MODEL_OUT_DIR / "shap_nearfreeze_vs_clearphase_ratio.csv")
print("  Saved: shap_nearfreeze_vs_clearphase_ratio.csv")

print("\nDone — all SHAP wet-bulb bin outputs written.")

In [ ]:
# =============================================================================
# EVALUATE, VISUALIZE, AND EXPORT
# =============================================================================
#
# Single self-contained cell replacing the previous cells 18, 19, and 20.
#
# Structure
# ---------
#   SECTION 0  — Imports and style constants
#   SECTION 1  — Load saved predictions (self-contained; re-runnable without
#                executing any earlier modelling cells)
#   SECTION 2  — Compute ALL metrics once into a flat dict (M)
#   SECTION 3  — Print console summary (reads from M, no recomputation)
#   SECTION 4  — Story 1: Binary discrimination  (confusion matrices + ROC/PR)
#   SECTION 5  — Story 2: Probability calibration (reliability diagram)
#   SECTION 6  — Story 3: Performance vs. T_wet   (F1 / mix-capture profiles)
#   SECTION 7  — Story 4: Uncertainty band placement
#   SECTION 8  — Story 5: Near-freezing deep-dive (3-class confusion matrices)
#   SECTION 9  — Save metrics_summary.json
#
# Every metric is computed exactly once in Section 2.
# All plots and exports read from M or the dataframes directly.
# =============================================================================


# =============================================================================
# SECTION 0 — IMPORTS AND STYLE
# =============================================================================

import datetime
import json

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)

PHASE_COLORS  = {"snow": "#3a86ff", "rain": "#2dc653", "mix": "#e377c2"}
SPLIT_COLORS  = {"val": "#8338ec",  "test": "#ff006e",
                 "Validation": "#8338ec", "Test": "#ff006e"}

plt.rcParams.update({
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.25,
    "font.size":         11,
})


# =============================================================================
# SECTION 1 — LOAD SAVED PREDICTIONS
# =============================================================================
# Re-loads from the parquet files saved by the export cell so this section is
# fully self-contained and can be re-run independently of the modelling cells.
# If df_val_fit / df_test_fit etc. are already in scope from Cell 14, they are
# used directly; the parquet reload is a safety fallback.

try:
    _ = df_val_fit
    print("Using in-memory dataframes from Cell 14.")
except NameError:
    print("Loading predictions from parquet …")
    df_val_fit   = pd.read_parquet(MODEL_OUT_DIR / "val_fit_binary_predictions_combined.parquet")
    df_test_fit  = pd.read_parquet(MODEL_OUT_DIR / "test_fit_binary_predictions_combined.parquet")
    df_val_full  = pd.read_parquet(MODEL_OUT_DIR / "val_full_uncertainty_predictions_combined.parquet")
    df_test_full = pd.read_parquet(MODEL_OUT_DIR / "test_full_uncertainty_predictions_combined.parquet")

# ── Reconstruct numpy arrays ──────────────────────────────────────────────────
y_val_bin  = df_val_fit["phase_binary"].to_numpy()
y_test_bin = df_test_fit["phase_binary"].to_numpy()

val_fit_prob_raw  = df_val_fit["p_snow_raw"].to_numpy()
val_fit_prob_cal  = df_val_fit["p_snow_cal"].to_numpy()
test_fit_prob_raw = df_test_fit["p_snow_raw"].to_numpy()
test_fit_prob_cal = df_test_fit["p_snow_cal"].to_numpy()

val_fit_pred_binary  = df_val_fit["prediction_binary05"].to_numpy()
test_fit_pred_binary = df_test_fit["prediction_binary05"].to_numpy()

y_val_fit_phase  = df_val_fit["phase_full"].to_numpy()
y_test_fit_phase = df_test_fit["phase_full"].to_numpy()

# Band parameters — prefer in-scope vars, fall back to stored constants
try:
    _base_hb, _extra_hb, _sigma = BEST_BASE_HB, BEST_EXTRA_HB, BEST_SIGMA
except NameError:
    _base_hb, _extra_hb, _sigma = BAND_BASE, BAND_EXTRA, BAND_SIGMA

print(f"  val_fit   : {df_val_fit.shape}")
print(f"  test_fit  : {df_test_fit.shape}")
print(f"  val_full  : {df_val_full.shape}")
print(f"  test_full : {df_test_full.shape}")


# =============================================================================
# SECTION 2 — COMPUTE ALL METRICS ONCE
# =============================================================================
# All metrics are stored in M.  Nothing below this section calls a sklearn
# metric function — they all read from M or the dataframes.

def _gaussian_half_band(t_wet, base=_base_hb, extra=_extra_hb, sigma=_sigma):
    return np.clip(
        base + extra * np.exp(-(np.asarray(t_wet, float) ** 2) / (2 * sigma ** 2)),
        0.0, 0.5,
    )

def _twet_performance_profile(df_full, bin_edges):
    """Per-bin F1 (snow, rain) and mix capture across T_wet bins."""
    records = []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        sub = df_full[(df_full["temp_wet"] >= lo) & (df_full["temp_wet"] < hi)]
        if len(sub) < 10:
            continue
        y_t = sub["phase_full"].to_numpy()
        y_p = sub["prediction_phase_uncertainty"].to_numpy()

        def _f1(code):
            tp = np.sum((y_t == code) & (y_p == code))
            fp = np.sum((y_t != code) & (y_p == code))
            fn = np.sum((y_t == code) & (y_p != code))
            pr = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            rc = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            return 2 * pr * rc / (pr + rc) if (pr + rc) > 0 else 0.0

        true_mix = y_t == MIX_CODE
        records.append({
            "t_mid":       (lo + hi) / 2,
            "n":           len(sub),
            "f1_snow":     _f1(SNOW_CODE),
            "f1_rain":     _f1(RAIN_CODE),
            "mix_capture": float(np.mean(y_p[true_mix] == MIX_CODE)) if true_mix.any() else np.nan,
            "accuracy":    accuracy_score(y_t, y_p),
        })
    return pd.DataFrame(records)

# ── Helper: scalar metric blocks ──────────────────────────────────────────────

def _prob_metrics(y_bin, p_raw, p_cal):
    return {
        "roc_auc_raw":       round(float(roc_auc_score(y_bin, p_raw)),               4),
        "roc_auc_cal":       round(float(roc_auc_score(y_bin, p_cal)),               4),
        "average_precision": round(float(average_precision_score(y_bin, p_cal)),     4),
        "brier_raw":         round(float(brier_score_loss(y_bin, p_raw)),            4),
        "brier_cal":         round(float(brier_score_loss(y_bin, p_cal)),            4),
        "log_loss_raw":      round(float(log_loss(y_bin, p_raw, labels=[0, 1])),     4),
        "log_loss_cal":      round(float(log_loss(y_bin, p_cal, labels=[0, 1])),     4),
    }

def _hard_metrics(y_true, y_pred, codes, names):
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=codes, zero_division=0)
    return {
        "accuracy":          round(float(accuracy_score(y_true, y_pred)),            4),
        "balanced_accuracy": round(float(balanced_accuracy_score(y_true, y_pred)),   4),
        "macro_f1":          round(float(f1_score(y_true, y_pred, average="macro",
                                                   zero_division=0)),                 4),
        "per_class": {
            n: {"precision": round(float(p[i]), 4),
                "recall":    round(float(r[i]), 4),
                "f1":        round(float(f[i]), 4),
                "support":   int(s[i])}
            for i, n in enumerate(names)
        },
    }

def _mix_metrics(y_true_ph, y_pred_ph):
    y_t, y_p       = np.asarray(y_true_ph), np.asarray(y_pred_ph)
    true_mix        = y_t == MIX_CODE
    true_pure       = np.isin(y_t, [SNOW_CODE, RAIN_CODE])
    pred_conf       = y_p != MIX_CODE
    mix_cap         = float(np.mean(y_p[true_mix] == MIX_CODE)) if true_mix.any() else None
    pure_cov        = float(np.mean(pred_conf[true_pure]))       if true_pure.any() else None
    cp_mask         = true_pure & pred_conf
    pure_acc        = float(np.mean(y_p[cp_mask] == y_t[cp_mask])) if cp_mask.any() else None
    conf_frac       = float(np.mean(pred_conf))
    return {
        "mix_capture_rate":              round(mix_cap,    4) if mix_cap    is not None else None,
        "pure_phase_confident_coverage": round(pure_cov,   4) if pure_cov   is not None else None,
        "pure_phase_confident_accuracy": round(pure_acc,   4) if pure_acc   is not None else None,
        "overall_confident_frac":        round(conf_frac,  4),
    }

def _nf_metrics(df_full, flag, codes, names):
    sub = df_full[df_full[flag]]
    if sub.empty:
        return {"n_obs": 0}
    p, r, f, s = precision_recall_fscore_support(
        sub["phase_full"], sub["prediction_phase_uncertainty"],
        labels=codes, zero_division=0)
    return {
        "n_obs":    int(len(sub)),
        "accuracy": round(float(accuracy_score(
            sub["phase_full"], sub["prediction_phase_uncertainty"])), 4),
        "per_class": {
            n: {"precision": round(float(p[i]), 4),
                "recall":    round(float(r[i]), 4),
                "f1":        round(float(f[i]), 4),
                "support":   int(s[i])}
            for i, n in enumerate(names)
        },
    }

# ── Assemble master metrics dict M ───────────────────────────────────────────
M = {
    "run_info": {
        "region":           REGION,
        "interp_type":      interp_type,
        "timestamp_utc":    datetime.datetime.utcnow().isoformat(timespec="seconds"),
        "scale_pos_weight": BEST_SCALE_POS_WEIGHT,
        "calibration":      "beta_calibration_regime_stratified",
        "gaussian_band": {
            "base_half_band":  _base_hb,
            "extra_half_band": _extra_hb,
            "sigma_degC":      _sigma,
        },
        "features": FEATURES,
        "n_train_snow": int((train_df_fit["phase_full"] == SNOW_CODE).sum()),
        "n_train_rain": int((train_df_fit["phase_full"] == RAIN_CODE).sum()),
        "n_val_snow":   int((val_df_fit["phase_full"]   == SNOW_CODE).sum()),
        "n_val_rain":   int((val_df_fit["phase_full"]   == RAIN_CODE).sum()),
        "n_test_snow":  int((test_df_fit["phase_full"]  == SNOW_CODE).sum()),
        "n_test_rain":  int((test_df_fit["phase_full"]  == RAIN_CODE).sum()),
    },
    "binary_probability": {
        "val":  _prob_metrics(y_val_bin,  val_fit_prob_raw,  val_fit_prob_cal),
        "test": _prob_metrics(y_test_bin, test_fit_prob_raw, test_fit_prob_cal),
    },
    "binary_hard_05": {
        "val":  _hard_metrics(y_val_fit_phase,  val_fit_pred_binary,
                              [SNOW_CODE, RAIN_CODE], ["snow", "rain"]),
        "test": _hard_metrics(y_test_fit_phase, test_fit_pred_binary,
                              [SNOW_CODE, RAIN_CODE], ["snow", "rain"]),
    },
    "uncertainty_3class": {
        "val":  _hard_metrics(df_val_full["phase_full"],
                              df_val_full["prediction_phase_uncertainty"],
                              [SNOW_CODE, RAIN_CODE, MIX_CODE], FULL_CLASS_NAMES),
        "test": _hard_metrics(df_test_full["phase_full"],
                              df_test_full["prediction_phase_uncertainty"],
                              [SNOW_CODE, RAIN_CODE, MIX_CODE], FULL_CLASS_NAMES),
    },
    "mix_behavior": {
        "val":  _mix_metrics(df_val_full["phase_full"],
                             df_val_full["prediction_phase_uncertainty"]),
        "test": _mix_metrics(df_test_full["phase_full"],
                             df_test_full["prediction_phase_uncertainty"]),
    },
    "near_freezing": {
        split_name: {
            flag: _nf_metrics(df_full, flag,
                               [SNOW_CODE, RAIN_CODE, MIX_CODE], FULL_CLASS_NAMES)
            for flag in ["near_freezing_air_2C", "near_freezing_wet_2C"]
        }
        for split_name, df_full in [("val", df_val_full), ("test", df_test_full)]
    },
}

# ── Pre-compute T_wet performance profiles (used by Story 3) ─────────────────
_BIN_EDGES   = np.arange(-6, 7, 1)
PROF = {
    "val":  _twet_performance_profile(df_val_full,  _BIN_EDGES),
    "test": _twet_performance_profile(df_test_full, _BIN_EDGES),
}

# ── Pre-compute band boundary arrays (used by Stories 2 & 4) ─────────────────
_t_grid        = np.linspace(-6, 6, 300)
_hb_grid       = _gaussian_half_band(_t_grid)
BAND_SNOW_LINE = 0.5 + _hb_grid   # upper boundary
BAND_RAIN_LINE = 0.5 - _hb_grid   # lower boundary
BAND_AT_ZERO   = float(_gaussian_half_band(np.array([0.0]))[0])

print("\n── Metrics computed ─────────────────────────────────────────────────────")
print(f"  Binary val  ROC AUC (cal)  : {M['binary_probability']['val']['roc_auc_cal']}")
print(f"  Binary test ROC AUC (cal)  : {M['binary_probability']['test']['roc_auc_cal']}")
print(f"  Binary val  log loss (cal) : {M['binary_probability']['val']['log_loss_cal']}")
print(f"  Binary test log loss (cal) : {M['binary_probability']['test']['log_loss_cal']}")
print(f"  3-class val  macro F1      : {M['uncertainty_3class']['val']['macro_f1']}")
print(f"  3-class test macro F1      : {M['uncertainty_3class']['test']['macro_f1']}")
print(f"  Val  mix capture rate      : {M['mix_behavior']['val']['mix_capture_rate']}")
print(f"  Test mix capture rate      : {M['mix_behavior']['test']['mix_capture_rate']}")


# =============================================================================
# SECTION 3 — CONSOLE SUMMARY (reads from M only)
# =============================================================================

SEP = "=" * 62

for split in ["val", "test"]:
    label = split.capitalize()

    # Probability metrics
    pm = M["binary_probability"][split]
    print(f"\n{SEP}")
    print(f"  {label.upper()} | PURE-PHASE BINARY PROBABILITY METRICS")
    print(SEP)
    print(f"  ROC AUC      raw / cal : {pm['roc_auc_raw']:.4f} / {pm['roc_auc_cal']:.4f}")
    print(f"  Average precision (cal): {pm['average_precision']:.4f}")
    print(f"  Brier score  raw / cal : {pm['brier_raw']:.4f} / {pm['brier_cal']:.4f}")
    print(f"  Log loss     raw / cal : {pm['log_loss_raw']:.4f} / {pm['log_loss_cal']:.4f}")

    # Hard binary
    hm = M["binary_hard_05"][split]
    print(f"\n{SEP}")
    print(f"  {label.upper()} | PURE-PHASE HARD BINARY (0.5 THRESHOLD)")
    print(SEP)
    print(f"  Accuracy          : {hm['accuracy']:.4f}")
    print(f"  Balanced accuracy : {hm['balanced_accuracy']:.4f}")
    print(f"  Macro F1          : {hm['macro_f1']:.4f}")
    for cls, v in hm["per_class"].items():
        print(f"    {cls:5s}  P={v['precision']:.3f}  R={v['recall']:.3f}  "
              f"F1={v['f1']:.3f}  n={v['support']}")

    # 3-class
    tm = M["uncertainty_3class"][split]
    print(f"\n{SEP}")
    print(f"  {label.upper()} | UNCERTAINTY-DERIVED 3-CLASS")
    print(SEP)
    print(f"  Accuracy          : {tm['accuracy']:.4f}")
    print(f"  Balanced accuracy : {tm['balanced_accuracy']:.4f}")
    print(f"  Macro F1          : {tm['macro_f1']:.4f}")
    for cls, v in tm["per_class"].items():
        print(f"    {cls:5s}  P={v['precision']:.3f}  R={v['recall']:.3f}  "
              f"F1={v['f1']:.3f}  n={v['support']}")

    # Mix behavior
    mb = M["mix_behavior"][split]
    print(f"\n{SEP}")
    print(f"  {label.upper()} | MIX / UNCERTAINTY BEHAVIOR")
    print(SEP)
    print(f"  Mix capture rate            : {mb['mix_capture_rate']:.4f}")
    print(f"  Pure-phase confident cov.   : {mb['pure_phase_confident_coverage']:.4f}")
    print(f"  Pure-phase confident acc.   : {mb['pure_phase_confident_accuracy']:.4f}")
    print(f"  Overall confident fraction  : {mb['overall_confident_frac']:.4f}")

    # Near-freezing
    for flag in ["near_freezing_air_2C", "near_freezing_wet_2C"]:
        nf = M["near_freezing"][split][flag]
        if nf.get("n_obs", 0) == 0:
            continue
        flag_label = "|T_air| ≤ 1°C" if "air" in flag else "|T_wet| ≤ 1°C"
        print(f"\n{SEP}")
        print(f"  {label.upper()} | {flag_label}  (n={nf['n_obs']})")
        print(SEP)
        for cls, v in nf["per_class"].items():
            print(f"    {cls:5s}  P={v['precision']:.3f}  R={v['recall']:.3f}  "
                  f"F1={v['f1']:.3f}  n={v['support']}")


# =============================================================================
# SECTION 4 — STORY 1: BINARY DISCRIMINATION
# =============================================================================
# Row 1: normalised confusion matrices (binary + 3-class) for val and test
# Row 2: ROC and PR curves, both splits on same axes

def _plot_norm_cm(ax, y_true, y_pred, labels, display_labels, title, cmap="Blues"):
    cm      = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    n       = len(labels)
    ax.imshow(cm_norm, vmin=0, vmax=1, cmap=cmap, aspect="auto")
    ax.set_xticks(range(n)); ax.set_xticklabels(display_labels)
    ax.set_yticks(range(n)); ax.set_yticklabels(display_labels)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title, fontsize=10)
    ax.grid(False)
    for i in range(n):
        for j in range(n):
            color = "white" if cm_norm[i, j] > 0.6 else "black"
            ax.text(j, i, f"{cm_norm[i,j]:.0%}\n({cm[i,j]})",
                    ha="center", va="center", fontsize=9, color=color)

fig = plt.figure(figsize=(16, 9))
fig.suptitle("Story 1 — Binary discrimination (pure-phase rain / snow)", fontsize=13, y=1.01)

ax00 = fig.add_subplot(2, 4, 1)
ax01 = fig.add_subplot(2, 4, 2)
ax02 = fig.add_subplot(2, 4, 3)
ax03 = fig.add_subplot(2, 4, 4)

_plot_norm_cm(ax00, y_val_fit_phase,  val_fit_pred_binary,
              [SNOW_CODE, RAIN_CODE], ["snow", "rain"], "Val — binary (0.5 thresh)")
_plot_norm_cm(ax01, y_test_fit_phase, test_fit_pred_binary,
              [SNOW_CODE, RAIN_CODE], ["snow", "rain"], "Test — binary (0.5 thresh)")
_plot_norm_cm(ax02, df_val_full["phase_full"],
              df_val_full["prediction_phase_uncertainty"],
              [SNOW_CODE, RAIN_CODE, MIX_CODE], FULL_CLASS_NAMES,
              "Val — 3-class (uncertainty band)")
_plot_norm_cm(ax03, df_test_full["phase_full"],
              df_test_full["prediction_phase_uncertainty"],
              [SNOW_CODE, RAIN_CODE, MIX_CODE], FULL_CLASS_NAMES,
              "Test — 3-class (uncertainty band)")

ax_roc = fig.add_subplot(2, 4, (5, 6))
ax_pr  = fig.add_subplot(2, 4, (7, 8))

for y_bin, p_cal, split in [
    (y_val_bin,  val_fit_prob_cal,  "val"),
    (y_test_bin, test_fit_prob_cal, "test"),
]:
    fpr, tpr, _  = roc_curve(y_bin, p_cal)
    prec, rec, _ = precision_recall_curve(y_bin, p_cal)
    auc = M["binary_probability"][split.lower()]["roc_auc_cal"]
    ap  = M["binary_probability"][split.lower()]["average_precision"]
    ax_roc.plot(fpr, tpr, color=SPLIT_COLORS[split], lw=2,
                label=f"{split}  AUC={auc:.3f}")
    ax_pr.plot(rec, prec, color=SPLIT_COLORS[split], lw=2,
               label=f"{split}  AP={ap:.3f}")

ax_roc.plot([0, 1], [0, 1], "--", color="grey", lw=1, alpha=0.6)
ax_roc.set(xlabel="FPR", ylabel="TPR", title="ROC — pure-phase")
ax_roc.legend(fontsize=10)
ax_pr.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall — pure-phase")
ax_pr.legend(fontsize=10)

plt.tight_layout()
plt.savefig(GRAPHICS_DIR / f"story1_discrimination_{interp_type}.png",
            dpi=180, bbox_inches="tight")
plt.show()


# =============================================================================
# SECTION 5 — STORY 2: PROBABILITY CALIBRATION
# =============================================================================
# Reliability diagram (quantile-binned) + p(snow) histogram below.
# Uncertainty band at T_wet=0 shown as a shaded region.

rain_thresh_0 = 0.5 - BAND_AT_ZERO
snow_thresh_0 = 0.5 + BAND_AT_ZERO

fig, axes = plt.subplots(2, 2, figsize=(12, 9),
                         gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle("Story 2 — Probability calibration (reliability diagram)", fontsize=13)

for col, (y_bin, p_raw, p_cal, split) in enumerate([
    (y_val_bin,  val_fit_prob_raw,  val_fit_prob_cal,  "val"),
    (y_test_bin, test_fit_prob_raw, test_fit_prob_cal, "test"),
]):
    ax_rel  = axes[0, col]
    ax_hist = axes[1, col]

    frac_raw, mean_raw = calibration_curve(y_bin, p_raw, n_bins=15, strategy="quantile")
    frac_cal, mean_cal = calibration_curve(y_bin, p_cal, n_bins=15, strategy="quantile")

    ax_rel.axvspan(rain_thresh_0, snow_thresh_0, alpha=0.10, color="orange",
                   label=f"Uncertainty band at T_wet=0°C  ({rain_thresh_0:.2f}–{snow_thresh_0:.2f})")
    ax_rel.plot([0, 1], [0, 1], "--", color="grey", lw=1.2, alpha=0.7,
                label="Perfect calibration")
    ax_rel.plot(mean_raw, frac_raw, "o--", color="#aaaaaa", lw=1.5, ms=5,
                label="Raw XGBoost")
    ax_rel.plot(mean_cal, frac_cal, "o-", color=SPLIT_COLORS[split], lw=2, ms=6,
                label="Calibrated (beta)")
    ax_rel.set(xlim=(0, 1), ylim=(0, 1), ylabel="Observed snow frequency",
               title=split)
    ax_rel.legend(fontsize=9)

    bs_r = M["binary_probability"][split.lower()]["brier_raw"]
    bs_c = M["binary_probability"][split.lower()]["brier_cal"]
    ll_c = M["binary_probability"][split.lower()]["log_loss_cal"]
    ax_rel.text(0.03, 0.92,
                f"Brier raw={bs_r:.3f}  cal={bs_c:.3f}  |  LogLoss cal={ll_c:.3f}",
                transform=ax_rel.transAxes, fontsize=8.5, color="dimgrey")

    ax_hist.hist(p_cal, bins=30, color=SPLIT_COLORS[split], alpha=0.7, edgecolor="none")
    ax_hist.axvspan(rain_thresh_0, snow_thresh_0, alpha=0.15, color="orange")
    ax_hist.set(xlabel="Calibrated p(snow)", ylabel="Count")
    ax_hist.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x)}"))

plt.tight_layout()
plt.savefig(GRAPHICS_DIR / f"story2_calibration_{interp_type}.png",
            dpi=180, bbox_inches="tight")
plt.show()


# =============================================================================
# SECTION 6 — STORY 3: PERFORMANCE VS. WET-BULB TEMPERATURE
# =============================================================================
# F1 (snow, rain) and mix capture rate in 1°C T_wet bins.
# Both splits on the same axes. Bubble size = n observations in bin.

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle("Story 3 — Performance vs. wet-bulb temperature (1°C bins)", fontsize=13)

_metrics_to_plot = [
    ("f1_snow",     f"F1 — snow",        PHASE_COLORS["snow"]),
    ("f1_rain",     f"F1 — rain",        PHASE_COLORS["rain"]),
    ("mix_capture", "Mix capture rate",  PHASE_COLORS["mix"]),
]

for ax, (col, ylabel, color) in zip(axes, _metrics_to_plot):
    for split_key, ls in [("val", "--"), ("test", "-")]:
        prof  = PROF[split_key]
        label = split_key.capitalize()
        ax.plot(prof["t_mid"], prof[col], ls, color=color, lw=2.2,
                label=label, marker="o", ms=5)
        ax.scatter(prof["t_mid"], prof[col],
                   s=prof["n"] / prof["n"].max() * 120,
                   color=color, alpha=0.25, edgecolors="none")

    ax.axvspan(-1, 1, alpha=0.08, color="orange", label="|T_wet| ≤ 1°C")
    ax.axvline(0, color="black", lw=0.8, alpha=0.4)
    ax.set(xlabel="Wet-bulb temperature (°C)", ylabel=ylabel, title=ylabel,
           xlim=(_BIN_EDGES[0], _BIN_EDGES[-1]), ylim=(0, 1.05))
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(GRAPHICS_DIR / f"story3_twet_performance_{interp_type}.png",
            dpi=180, bbox_inches="tight")
plt.show()


# =============================================================================
# SECTION 7 — STORY 4: UNCERTAINTY BAND PLACEMENT
# =============================================================================
# Panel A — True-mix observations coloured by band capture (val + test combined)
# Panel B — p(snow) violin distributions by true phase across 2°C T_wet bins

# ── Combine val + test mix observations ──────────────────────────────────────
df_mix_all = pd.concat([
    df_val_full[df_val_full["phase_full"] == MIX_CODE].assign(split="Validation"),
    df_test_full[df_test_full["phase_full"] == MIX_CODE].assign(split="Test"),
], ignore_index=True)

hb_per_obs    = _gaussian_half_band(df_mix_all["temp_wet"].to_numpy())
inside_band   = (
    (df_mix_all["p_snow_cal"].to_numpy() > 0.5 - hb_per_obs) &
    (df_mix_all["p_snow_cal"].to_numpy() < 0.5 + hb_per_obs)
)
df_mix_all["inside_band"] = inside_band
n_inside      = int(inside_band.sum())
n_total_mix   = len(inside_band)
capture_pct   = 100 * n_inside / n_total_mix if n_total_mix > 0 else 0

# ── Panel B: violins ──────────────────────────────────────────────────────────
VIOLIN_BINS   = [(-6, -4), (-4, -2), (-2, 0), (0, 2), (2, 4), (4, 6)]
bin_labels    = [f"{lo}–{hi}" for lo, hi in VIOLIN_BINS]
df_full_combo = pd.concat([df_val_full, df_test_full], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(17, 7))
fig.suptitle("Story 4 — Uncertainty band placement", fontsize=13, y=1.01)

# Panel A ─────────────────────────────────────────────────────────────────────
ax_a = axes[0]
ax_a.fill_between(_t_grid, BAND_RAIN_LINE, BAND_SNOW_LINE,
                  alpha=0.10, color="orange", zorder=1)
ax_a.plot(_t_grid, BAND_SNOW_LINE, color="black", lw=2.0, ls="-",  zorder=2,
          label="Snow threshold")
ax_a.plot(_t_grid, BAND_RAIN_LINE, color="black", lw=2.0, ls="--", zorder=2,
          label="Rain threshold")

for flag, label, color, marker, zorder in [
    (False, "Missed by band",   "#cc3311", "x", 3),
    (True,  "Captured by band", "#009988", "o", 4),
]:
    sub = df_mix_all[df_mix_all["inside_band"] == flag]
    ax_a.scatter(sub["temp_wet"], sub["p_snow_cal"],
                 c=color, marker=marker, s=55, alpha=0.75,
                 linewidths=1.2, label=f"{label}  (n={len(sub)})", zorder=zorder)

ax_a.axvline(0,   color="grey", lw=0.8, alpha=0.45)
ax_a.axhline(0.5, color="grey", lw=0.8, alpha=0.45)
ax_a.set(xlim=(-6, 6), ylim=(-0.04, 1.04),
         xlabel="Wet-bulb temperature (°C)", ylabel="Calibrated p(snow)")
ax_a.set_title(
    f"A — True-mix observations: band capture\n"
    f"(val + test combined, n={n_total_mix},  {capture_pct:.0f}% inside band)",
    fontsize=11)
ax_a.legend(fontsize=9, loc="upper right")

# Panel B ─────────────────────────────────────────────────────────────────────
ax_b     = axes[1]
n_phases = 3
group_w  = 0.8
phase_w  = group_w / n_phases

for b_idx, (lo, hi) in enumerate(VIOLIN_BINS):
    mask_bin = (df_full_combo["temp_wet"] >= lo) & (df_full_combo["temp_wet"] < hi)
    sub_bin  = df_full_combo[mask_bin]
    for p_idx, (code, cname) in enumerate(
        zip([SNOW_CODE, RAIN_CODE, MIX_CODE], FULL_CLASS_NAMES)
    ):
        vals  = sub_bin.loc[sub_bin["phase_full"] == code, "p_snow_cal"].to_numpy()
        x_pos = b_idx + (p_idx - 1) * phase_w
        if len(vals) < 4:
            if len(vals) > 0:
                ax_b.plot(x_pos, float(np.median(vals)), "_",
                          color=PHASE_COLORS[cname], ms=12, mew=2)
            continue
        parts = ax_b.violinplot(vals, positions=[x_pos], widths=phase_w * 0.85,
                                showmedians=True, showextrema=False)
        for pc in parts["bodies"]:
            pc.set_facecolor(PHASE_COLORS[cname])
            pc.set_edgecolor("none")
            pc.set_alpha(0.70)
        parts["cmedians"].set_color("black")
        parts["cmedians"].set_linewidth(1.5)

ax_b.axhspan(float(BAND_RAIN_LINE.min()), float(BAND_SNOW_LINE.max()),
             alpha=0.06, color="orange", label="Max band extent")
ax_b.axhline(0.5, color="grey", lw=0.8, alpha=0.45)
ax_b.set_xticks(range(len(VIOLIN_BINS)))
ax_b.set_xticklabels(bin_labels, fontsize=9)
ax_b.set(xlabel="Wet-bulb temperature bin (°C)", ylabel="Calibrated p(snow)",
         ylim=(-0.04, 1.04))
ax_b.set_title("B — p(snow) distributions by true phase\n(val + test combined, 2°C bins)",
               fontsize=11)
ax_b.legend(handles=[
    Patch(facecolor=PHASE_COLORS[c], label=c, alpha=0.75) for c in FULL_CLASS_NAMES
] + [Patch(facecolor="orange", alpha=0.20, label="Max band extent")],
    fontsize=9, loc="upper right")

plt.tight_layout()
plt.savefig(GRAPHICS_DIR / f"story4_band_placement_{interp_type}.png",
            dpi=180, bbox_inches="tight")
plt.show()

print(f"\nStory 4 — Mix band capture (val + test combined)")
print(f"  Inside band  : {n_inside} / {n_total_mix}  ({capture_pct:.1f}%)")
print(f"  Outside band : {n_total_mix - n_inside}  ({100 - capture_pct:.1f}%)")


# =============================================================================
# SECTION 8 — STORY 5: NEAR-FREEZING DEEP-DIVE
# =============================================================================
# 2 × 2 grid of normalised 3-class confusion matrices
#   rows → val / test    cols → |T_air| ≤ 1°C / |T_wet| ≤ 1°C

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
fig.suptitle("Story 5 — Near-freezing regime: 3-class performance deep-dive",
             fontsize=13)

for (r, c), (df_full, split_key, flag) in zip(
    [(0, 0), (0, 1), (1, 0), (1, 1)],
    [
        (df_val_full,  "val",  "near_freezing_air_2C"),
        (df_val_full,  "val",  "near_freezing_wet_2C"),
        (df_test_full, "test", "near_freezing_air_2C"),
        (df_test_full, "test", "near_freezing_wet_2C"),
    ],
):
    sub        = df_full[df_full[flag]]
    flag_label = "|T_air| ≤ 2°C" if "air" in flag else "|T_wet| ≤ 2°C"
    split_label = split_key.capitalize()
    ax         = axes[r, c]

    if sub.empty:
        ax.set_visible(False)
        continue

    _plot_norm_cm(
        ax,
        sub["phase_full"],
        sub["prediction_phase_uncertainty"],
        [SNOW_CODE, RAIN_CODE, MIX_CODE],
        FULL_CLASS_NAMES,
        f"{split_label} — {flag_label}  (n={len(sub)})",
    )

plt.tight_layout()
plt.savefig(GRAPHICS_DIR / f"story5_near_freezing_{interp_type}.png",
            dpi=180, bbox_inches="tight")
plt.show()


# =============================================================================
# SECTION 9 — SAVE metrics_summary.json
# =============================================================================

_out = MODEL_OUT_DIR / "metrics_summary.json"
with open(_out, "w") as f:
    json.dump(M, f, indent=2)

print(f"\nSaved: {_out.name}")
print(f"All plots saved to: {GRAPHICS_DIR}")